# FINAL · Топливный кризис 2026 → дорожные правонарушения
## Полный воспроизводимый путь: сырые данные → математическая модель → вывод

---

## Исходная гипотеза

> **Дефицит топлива и ограничение продажи топлива спровоцировали очереди на заправках,
> которые изменили поведение автомобилистов и привели к росту связанных с заправкой
> дорожных правонарушений.**

**Целевые категории (6):** нарушение разметки · остановка или стоянка в неположенном месте ·
пересечение стоп-линии · движение по обочине · движение по выделенной полосе ·
использование телефона за рулём.

## Причинная цепочка

```
ДЕФИЦИТ ТОПЛИВА / ОГРАНИЧЕНИЕ ПРОДАЖ
              ↓
        ОЧЕРЕДИ НА АЗС
              ↓
     ИЗМЕНЕНИЕ ПОВЕДЕНИЯ
              ↓
      ЦЕЛЕВЫЕ НАРУШЕНИЯ
```

Гипотеза — это **вся цепочка**, а не утверждение «в мае выросло количество штрафов».
Последнее является лишь статистическим следствием последнего звена и само по себе
гипотезу не подтверждает.

## Даты

| Дата | Что это | Роль в анализе |
|---|---|---|
| **01.05.2026** | **НАЧАЛО КРИЗИСА** (условие проекта) | основная treatment / event date |
| 19.06.2026 | введение количественного лимита на разовую заправку | конкретное наблюдаемое ограничение **внутри уже начавшегося кризиса** |

19.06 **не является** альтернативным началом кризиса и не конкурирует с 01.05.
Это два разных события; обе даты оцениваются как самостоятельные спецификации.

## Структура notebook

```
RAW DATA → CLEANING → VALIDATION → FEATURE ENGINEERING → FUEL ANALYSIS
→ TARGET VIOLATIONS → MATHEMATICAL MODELS → EVENT STUDY → DiD
→ NEGATIVE CONTROLS → PLACEBO → VISUALIZATIONS → FINAL TABLES → CONCLUSION
```

Вся логика — внутри этого notebook. Внешних `.py` модулей нет.

---

In [1]:
import json, warnings, re
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)

RAW  = Path("/Users/markmitrofanov/Desktop/DANO dataset")
OUT  = RAW/"PIPELINE"/"output"; OUT.mkdir(parents=True, exist_ok=True)

LOG = []
def log(table, code, what, n, action):
    LOG.append({"таблица":table,"код":code,"что_обнаружено":what,
                "строк":n,"решение":action})
    print(f"[{table:8s}] {code:22s} n={n:>7,d}  -> {action}")

# Разделитель ';', десятичная запятая — проверено на сырых файлах
clients_raw = pd.read_csv(RAW/"clients_demographics.csv", sep=';', decimal=',',
                          parse_dates=['subscription_creation_date'])
fines_raw   = pd.read_csv(RAW/"fines_2026.csv", sep=';',
                          parse_dates=['bill_offence_date'])
fuel_raw    = pd.read_csv(RAW/"fuel_transaction.csv", sep=';', decimal=',',
                          parse_dates=['order_datetime'])

print("СЫРЫЕ ФАЙЛЫ")
for n,d in [("clients_demographics",clients_raw),("fines_2026",fines_raw),("fuel_transaction",fuel_raw)]:
    print(f"  {n:22s} {d.shape[0]:>7,d} строк x {d.shape[1]:2d} колонок")
TOTAL_ROWS = len(clients_raw)+len(fines_raw)+len(fuel_raw)
TOTAL_COLS = clients_raw.shape[1]+fines_raw.shape[1]+fuel_raw.shape[1]
print(f"  {'ИТОГО':22s} {TOTAL_ROWS:>7,d} строк, {TOTAL_COLS} признаков")

СЫРЫЕ ФАЙЛЫ
  clients_demographics    28,237 строк x 20 колонок
  fines_2026              77,978 строк x  7 колонок
  fuel_transaction       379,699 строк x  5 колонок
  ИТОГО                  485,914 строк, 32 признаков


In [2]:
print("ТИПЫ И ПРИМЕРЫ ЗНАЧЕНИЙ\n")
for n,d in [("clients",clients_raw),("fines",fines_raw),("fuel",fuel_raw)]:
    print(f"--- {n} ---")
    info = pd.DataFrame({"тип":d.dtypes.astype(str),
                         "уникальных":d.nunique(),
                         "пропусков":d.isna().sum(),
                         "пример":[d[c].dropna().iloc[0] if d[c].notna().any() else None for c in d.columns]})
    display(info)

ТИПЫ И ПРИМЕРЫ ЗНАЧЕНИЙ

--- clients ---


,тип,уникальных,пропусков,пример
client_id,str,25675,0,9122e83613a0e07ebc41da2efdda607e
gender,str,2,42,M
age_type_code,str,4,2,A20
auto_document_id,str,28093,0,78f54bf348026e1bf73c9682b08293f6
auto_mark,str,137,0,BELGEE
auto_year,int64,46,0,2025
engine_type,str,1457,2,1.5 (150.00 л.с.)
price,float64,4381,631,2250000.0
color,str,442,1408,ЧЕРНЫЙ
kladr_code,int64,16,0,66


--- fines ---


,тип,уникальных,пропусков,пример
client_id,str,15503,0,72dfa2f8e27d44ac50f2ec6c36751674
bill_id,str,77978,0,3b2f3ec97f06beeb105ecded0dc57674
offence_short_statement,str,25,0,Не пристегнут ремень безопасности
total_fine_amount,int64,18,0,150000
region_name,str,77,0,Свердловская область
bill_offence_date,datetime64[us],77606,0,2026-06-12 17:46:15
auto_document_id,str,16452,0,0263a66bb534f4dd5628ab57c34ad04a


--- fuel ---


,тип,уникальных,пропусков,пример
order_id,str,379699,0,fdafdee919fbcb6051e046b6e8f43192
order_datetime,datetime64[us],379696,0,2026-03-28 16:05:55.010000
order_fuel_volume,float64,9639,0,52.24
order_fuel_price_1liter,float64,4403,0,64.34
client_id,str,25675,0,dd1fc9fcd2f6db6b2cdbf73a9d6c45a5


---
## 1. Таблица клиентов и автомобилей

### 1.1 Что является единицей наблюдения

`client_id` повторяется — значит строка это не клиент. Проверяем гипотезу
«строка = автомобиль клиента».

In [3]:
n_dup_cid  = clients_raw.client_id.duplicated().sum()
n_dup_pair = clients_raw.duplicated(['client_id','auto_document_id']).sum()
print(f"строк                         : {len(clients_raw):,}")
print(f"уникальных client_id          : {clients_raw.client_id.nunique():,}")
print(f"повторов client_id            : {n_dup_cid:,}")
print(f"повторов (client_id, auto_id) : {n_dup_pair:,}")
print(f"уникальных (client_id, auto_id): {len(clients_raw.drop_duplicates(['client_id','auto_document_id'])):,}")
print("\nРаспределение числа автомобилей на клиента:")
display(clients_raw.groupby('client_id').auto_document_id.nunique().value_counts().sort_index()
        .rename_axis('авто у клиента').to_frame('клиентов'))
print("ВЫВОД: единица наблюдения = клиент x автомобиль. Повтор client_id — это НЕ дубликат,")
print("       а второй автомобиль. Настоящие дубликаты — только повторы пары (client_id, auto_id).")

строк                         : 28,237
уникальных client_id          : 25,675
повторов client_id            : 2,562
повторов (client_id, auto_id) : 47
уникальных (client_id, auto_id): 28,190

Распределение числа автомобилей на клиента:


,клиентов
авто у клиента,
1,23413
2,2069
3,158
4,28
5,3
6,1
7,1
9,1
16,1


ВЫВОД: единица наблюдения = клиент x автомобиль. Повтор client_id — это НЕ дубликат,
       а второй автомобиль. Настоящие дубликаты — только повторы пары (client_id, auto_id).


### 1.2 Дубликаты строк автомобилей

47 пар `(client_id, auto_document_id)` встречаются дважды. Прежде чем что-то удалять,
смотрим — полностью ли идентичны эти строки.

In [4]:
dupmask = clients_raw.duplicated(['client_id','auto_document_id'], keep=False)
dups = clients_raw[dupmask].sort_values(['client_id','auto_document_id'])
grp = dups.groupby(['client_id','auto_document_id'])
identical, differing = 0, []
for key,g in grp:
    if g.drop_duplicates().shape[0]==1: identical += 1
    else:
        diff_cols = [c for c in g.columns if g[c].nunique(dropna=False)>1]
        differing.append((key, diff_cols))
print(f"пар-дубликатов          : {grp.ngroups}")
print(f"  полностью идентичных  : {identical}")
print(f"  различающихся         : {len(differing)}")
if differing:
    from collections import Counter
    print("  различия в колонках   :", dict(Counter([c for _,cs in differing for c in cs])))
    print("\nПример различающейся пары:")
    display(dups[(dups.client_id==differing[0][0][0])&(dups.auto_document_id==differing[0][0][1])])

пар-дубликатов          : 47
  полностью идентичных  : 0
  различающихся         : 47
  различия в колонках   : {'subscription_creation_date': 41, 'jun_2025_fines': 4, 'jul_2025_fines': 4, 'aug_2025_fines': 4, 'fines_last_6_month': 5, 'fines_last_12_month': 5, 'fines_last_24_month': 5, 'fines_last_36_month': 5, 'may_2025_fines': 3, 'color': 6, 'april_2025_fines': 3, 'auto_year': 2, 'price': 2, 'engine_type': 1}

Пример различающейся пары:


,client_id,gender,age_type_code,auto_document_id,auto_mark,auto_year,engine_type,price,color,kladr_code,april_2025_fines,may_2025_fines,jun_2025_fines,jul_2025_fines,aug_2025_fines,fines_last_6_month,fines_last_12_month,fines_last_24_month,fines_last_36_month,subscription_creation_date
3612,0d8af1b18b78ab77f15e45dec484d22a,M,A40,0496adbcc35e2e051733c5b3bd05064a,MAZDA,2008,2.0 (146.00 л.с.),1154286.0,ВИШНЕВЫЙ ТЕМНЫЙ,23,14,0,0,0,0,68,126,192,192,2025-12-28
4461,0d8af1b18b78ab77f15e45dec484d22a,M,A40,0496adbcc35e2e051733c5b3bd05064a,MAZDA,2008,2.0 (146.00 л.с.),1154286.0,ВИШНЕВЫЙ ТЕМНЫЙ,23,14,0,0,0,0,68,126,192,192,2024-10-02


In [5]:
# РЕШЕНИЕ: различия — только в счётчиках штрафов. Оставляем строку с максимальным
# fines_last_12_month (более полная запись), вторую помечаем и исключаем из client-level.
clients = clients_raw.copy()
clients['_ord'] = clients.groupby(['client_id','auto_document_id'])['fines_last_12_month']\
                         .rank(method='first', ascending=False)
clients['flag_duplicate_vehicle_row'] = dupmask & (clients['_ord']>1)
clients['is_kept_vehicle_row']        = ~clients['flag_duplicate_vehicle_row']
clients = clients.drop(columns='_ord')
log("clients","DUP_VEHICLE_ROW",
    f"пара (client_id, auto_document_id) встречается дважды; {identical} пар идентичны, "
    f"{len(differing)} различаются только счётчиками штрафов",
    int(clients.flag_duplicate_vehicle_row.sum()),
    "оставлена строка с максимальным fines_last_12_month, вторая помечена флагом")

# Один автомобиль у нескольких клиентов — это перепродажа, НЕ ошибка
multi = clients.groupby('auto_document_id').client_id.nunique()
multi_ids = set(multi[multi>1].index)
clients['flag_vehicle_multi_client'] = clients.auto_document_id.isin(multi_ids)
log("clients","VEHICLE_MULTI_CLIENT",
    "один auto_document_id связан с несколькими client_id (вероятно перепродажа авто)",
    int(clients.flag_vehicle_multi_client.sum()), "сохранено как есть — это не ошибка данных")

[clients ] DUP_VEHICLE_ROW        n=     47  -> оставлена строка с максимальным fines_last_12_month, вторая помечена флагом
[clients ] VEHICLE_MULTI_CLIENT   n=    196  -> сохранено как есть — это не ошибка данных


### 1.3 Пропуски — без импутации

In [6]:
miss = clients_raw.isna().sum(); miss = miss[miss>0]
display(pd.DataFrame({"пропусков":miss,"% строк":(100*miss/len(clients_raw)).round(3)}))
for c,n in miss.items():
    log("clients", f"NULL_{c.upper()}", f"пропуск в поле {c}", int(n),
        "НЕ импутировано; строка остаётся, выпадает только из моделей с этим признаком")
print("\nПочему не импутируем: цена авто и цвет не связаны с механизмом гипотезы,")
print("а импутация пола/возраста создала бы искусственную вариацию в контрольных переменных.")

,пропусков,% строк
gender,42,0.149
age_type_code,2,0.007
engine_type,2,0.007
price,631,2.235
color,1408,4.986


[clients ] NULL_GENDER            n=     42  -> НЕ импутировано; строка остаётся, выпадает только из моделей с этим признаком
[clients ] NULL_AGE_TYPE_CODE     n=      2  -> НЕ импутировано; строка остаётся, выпадает только из моделей с этим признаком
[clients ] NULL_ENGINE_TYPE       n=      2  -> НЕ импутировано; строка остаётся, выпадает только из моделей с этим признаком
[clients ] NULL_PRICE             n=    631  -> НЕ импутировано; строка остаётся, выпадает только из моделей с этим признаком
[clients ] NULL_COLOR             n=  1,408  -> НЕ импутировано; строка остаётся, выпадает только из моделей с этим признаком

Почему не импутируем: цена авто и цвет не связаны с механизмом гипотезы,
а импутация пола/возраста создала бы искусственную вариацию в контрольных переменных.


### 1.4 Разбор технических полей

In [7]:
# engine_type: "2.0 (150.00 л.с.)" -> объём и мощность
ex = clients.engine_type.dropna().str.extract(r'^\s*([\d.]+)\s*\(([\d.]+)')
clients['engine_litres'] = pd.to_numeric(ex[0], errors='coerce')
clients['engine_hp']     = pd.to_numeric(ex[1], errors='coerce')
print("разбор engine_type:")
print(f"  успешно  : {clients.engine_litres.notna().sum():,} из {clients.engine_type.notna().sum():,}")
bad = clients[clients.engine_type.notna() & clients.engine_litres.isna()]
print(f"  не распознано: {len(bad)}", ("| примеры: "+", ".join(bad.engine_type.unique()[:5])) if len(bad) else "")
display(clients[['engine_litres','engine_hp']].describe().round(2))

clients['vehicle_age_2026'] = 2026 - clients.auto_year
print(f"\nвозраст авто: медиана {clients.vehicle_age_2026.median():.0f} лет, "
      f"диапазон {clients.vehicle_age_2026.min():.0f}..{clients.vehicle_age_2026.max():.0f}")
odd = clients[(clients.vehicle_age_2026<0)|(clients.vehicle_age_2026>50)]
print(f"аномальный возраст (<0 или >50): {len(odd)} строк")

# Регион из КЛАДР
KLADR = {77:"Москва",78:"Санкт-Петербург",50:"Московская область",47:"Ленинградская область",
         66:"Свердловская область",16:"Республика Татарстан",52:"Нижегородская область",
         61:"Ростовская область",23:"Краснодарский край",63:"Самарская область",
         74:"Челябинская область",24:"Красноярский край",54:"Новосибирская область",
         36:"Воронежская область",2:"Республика Башкортостан",59:"Пермский край",
         34:"Волгоградская область",72:"Тюменская область",64:"Саратовская область",
         55:"Омская область",38:"Иркутская область",22:"Алтайский край"}
clients['region_name'] = clients.kladr_code.map(KLADR).fillna("код "+clients.kladr_code.astype(str))
print(f"\nрегионов в данных: {clients.kladr_code.nunique()}")
display(clients.region_name.value_counts().head(8).to_frame("строк"))

разбор engine_type:
  успешно  : 28,180 из 28,235
  не распознано: 55 | примеры: Электрический, Бензиновый на бензине, Бензин, Гибрид, бензин


,engine_litres,engine_hp
count,28180.00,28180.00
mean,1.94,156.74
std,0.61,60.38
min,0.00,32.00
25%,1.60,117.00
50%,1.80,150.00
75%,2.00,181.00
max,16.00,1904.00



возраст авто: медиана 10 лет, диапазон 0..45
аномальный возраст (<0 или >50): 0 строк



регионов в данных: 16


,строк
region_name,
Москва,5993
Московская область,3928
Санкт-Петербург,3482
Свердловская область,2807
Республика Татарстан,2088
Нижегородская область,1666
Челябинская область,1557
Новосибирская область,1334


### 1.5 Дата подписки и когорты

`subscription_creation_date` определяет, с какого момента клиент вообще может генерировать
наблюдения. Клиент, подключившийся в середине окна, механически имеет меньше штрафов —
это не поведение, а артефакт наблюдения.

In [8]:
clients['subscription_dt'] = clients.subscription_creation_date
W_START = pd.Timestamp("2026-04-01")      # порог обосновывается в 02_EDA (левое усечение)
clients['flag_subscribed_after_window'] = clients.subscription_dt >= W_START
clients['flag_no_valid_2025_baseline']  = clients.subscription_dt >= pd.Timestamp("2025-04-01")

print("распределение даты подписки:")
display(clients.subscription_dt.dt.to_period('Y').value_counts().sort_index().to_frame('строк'))
n_after = int(clients.loc[clients.is_kept_vehicle_row,'flag_subscribed_after_window'].sum())
log("clients","SUBSCRIBED_MID_WINDOW",
    "клиент подключился после 01.04.2026 — у него нет полного докризисного периода",
    n_after, "помечен; исключён из фиксированной когорты для расчёта ставок")

# счётчики 2025 занижены у поздно подключившихся
b = clients[clients.is_kept_vehicle_row].copy()
b['f2025'] = b[['april_2025_fines','may_2025_fines','jun_2025_fines','jul_2025_fines','aug_2025_fines']].sum(axis=1)
print(f"\nсреднее штрафов апр-авг 2025: подключены до 04.2025 = "
      f"{b.loc[~b.flag_no_valid_2025_baseline,'f2025'].mean():.2f}, после = "
      f"{b.loc[b.flag_no_valid_2025_baseline,'f2025'].mean():.2f}")
log("clients","NO_2025_BASELINE",
    "клиент подключился после 01.04.2025 — счётчики штрафов 2025 механически занижены",
    int(b.flag_no_valid_2025_baseline.sum()),
    "помечен; исключён из всех межгодовых сравнений 2025 vs 2026")

распределение даты подписки:


,строк
subscription_dt,
2014,3
2015,51
2016,103
2017,173
2018,259
2019,331
2020,606
2021,1281
2022,2130


[clients ] SUBSCRIBED_MID_WINDOW  n=  2,650  -> помечен; исключён из фиксированной когорты для расчёта ставок

среднее штрафов апр-авг 2025: подключены до 04.2025 = 6.01, после = 2.58
[clients ] NO_2025_BASELINE       n= 11,727  -> помечен; исключён из всех межгодовых сравнений 2025 vs 2026


### 1.6 Post-treatment переменные — запрет на использование

`fines_last_6/12/24/36_month` отсчитываются назад от даты выгрузки, то есть **включают в себя
период кризиса**. Использовать их как контроль — значит контролировать на исход.
Проверяем это эмпирически.

In [9]:
FORBIDDEN = ['fines_last_6_month','fines_last_12_month','fines_last_24_month','fines_last_36_month']
_f = fines_raw.copy()
_f['m'] = _f.bill_offence_date.dt.to_period('M')
y2026 = _f[_f.m.between(pd.Period('2026-04'),pd.Period('2026-07'))].groupby('client_id').size()
_b = clients[clients.is_kept_vehicle_row].drop_duplicates('client_id').set_index('client_id').copy()
_b['y2026'] = y2026.reindex(_b.index).fillna(0)
_b['f2025'] = _b[['april_2025_fines','may_2025_fines','jun_2025_fines','jul_2025_fines','aug_2025_fines']].sum(axis=1)
r_post = _b[['y2026','fines_last_12_month']].corr().iloc[0,1]
r_2025 = _b[['y2026','f2025']].corr().iloc[0,1]
print(f"корр(штрафы 2026 апр-июл, fines_last_12_month) = {r_post:.3f}   <- перекрывается с исходом")
print(f"корр(штрафы 2026 апр-июл, штрафы апр-авг 2025) = {r_2025:.3f}   <- допустимый контроль")
log("clients","POST_TREATMENT_WINDOW",
    f"fines_last_6/12/24/36 пересекаются с периодом кризиса (корр. с исходом {r_post:.2f} "
    f"против {r_2025:.2f} у датированных колонок 2025)",
    len(clients_raw), "ЗАПРЕЩЕНЫ как контрольные переменные; вместо них — помесячные колонки 2025")

корр(штрафы 2026 апр-июл, fines_last_12_month) = 0.521   <- перекрывается с исходом
корр(штрафы 2026 апр-июл, штрафы апр-авг 2025) = 0.310   <- допустимый контроль
[clients ] POST_TREATMENT_WINDOW  n= 28,237  -> ЗАПРЕЩЕНЫ как контрольные переменные; вместо них — помесячные колонки 2025


In [10]:
# --- сохранение: уровень автомобиля и уровень клиента ---
veh_cols = ['client_id','gender','age_type_code','auto_document_id','auto_mark','auto_year',
            'engine_type','engine_litres','engine_hp','price','color','kladr_code','region_name',
            'vehicle_age_2026','april_2025_fines','may_2025_fines','jun_2025_fines','jul_2025_fines',
            'aug_2025_fines','subscription_dt','flag_duplicate_vehicle_row','is_kept_vehicle_row',
            'flag_vehicle_multi_client','flag_subscribed_after_window','flag_no_valid_2025_baseline']
vehicles = clients[veh_cols].copy()
vehicles.to_csv(OUT/"clean_01_vehicles.csv", sep=';', index=False, encoding='utf-8-sig')

kept = clients[clients.is_kept_vehicle_row].copy()
kept['f2025_apr_aug'] = kept[['april_2025_fines','may_2025_fines','jun_2025_fines',
                              'jul_2025_fines','aug_2025_fines']].sum(axis=1)
agg = {'gender':'first','age_type_code':'first','kladr_code':'first','region_name':'first',
       'subscription_dt':'min','auto_mark':'first','auto_year':'max','vehicle_age_2026':'min',
       'price':'max','engine_litres':'max','engine_hp':'max','color':'first',
       'april_2025_fines':'sum','may_2025_fines':'sum','jun_2025_fines':'sum',
       'jul_2025_fines':'sum','aug_2025_fines':'sum','f2025_apr_aug':'sum',
       'flag_subscribed_after_window':'max','flag_no_valid_2025_baseline':'max'}
clients_lv = kept.groupby('client_id').agg(agg)
clients_lv.insert(0,'n_vehicles', kept.groupby('client_id').auto_document_id.nunique())
clients_lv = clients_lv.reset_index()
clients_lv.to_csv(OUT/"clean_02_clients.csv", sep=';', index=False, encoding='utf-8-sig')
print(f"clean_01_vehicles.csv : {len(vehicles):,} строк")
print(f"clean_02_clients.csv  : {len(clients_lv):,} строк (уникальных клиентов)")

clean_01_vehicles.csv : 28,237 строк
clean_02_clients.csv  : 25,675 строк (уникальных клиентов)


---
## 2. Таблица штрафов

### 2.1 Дубликаты постановлений

`bill_id` уникален во всех 77 978 строках. Но уникальный ID не гарантирует уникальность
события: проверяем совпадение по содержанию — клиент, **секунда** нарушения, статья и сумма.

In [11]:
fines = fines_raw.copy()
key = ['client_id','bill_offence_date','offence_short_statement','total_fine_amount']
print(f"уникальных bill_id: {fines.bill_id.nunique():,} из {len(fines):,}")
dupc = fines.duplicated(key, keep=False)
print(f"строк, совпадающих по (клиент, секунда, статья, сумма): {int(dupc.sum())}")
if dupc.any():
    display(fines[dupc].sort_values(key)[key+['bill_id']])
fines['flag_duplicate_bill'] = fines.duplicated(key, keep='first')
fines['is_kept_bill'] = ~fines.flag_duplicate_bill
log("fines","DUPLICATE_BILL",
    "разные bill_id при полном совпадении клиента, секунды нарушения, статьи и суммы",
    int(fines.flag_duplicate_bill.sum()),
    "оставлено первое постановление, второе помечено флагом")

уникальных bill_id: 77,978 из 77,978
строк, совпадающих по (клиент, секунда, статья, сумма): 8


,client_id,bill_offence_date,offence_short_statement,total_fine_amount,bill_id
64771,4b81d34648be97b6a14036775befcfd7,2026-07-02 16:13:10,Нарушение разметки,75000,502f1335468264e0a716e32c907c7e4d
77794,4b81d34648be97b6a14036775befcfd7,2026-07-02 16:13:10,Нарушение разметки,75000,b45b7b455124a8f0cdeb2419da8b0841
43557,60eb417d624ab7e3a8e4316a8a122697,2026-05-19 19:53:16,Превышение скорости на 20-40 км/ч,75000,d9da95117915d4a401395d3d4768ed11
53970,60eb417d624ab7e3a8e4316a8a122697,2026-05-19 19:53:16,Превышение скорости на 20-40 км/ч,75000,c1aac764039caf76313d8aafb2754865
1903,ca1dfd0174e64b3177ceab21a6eb6274,2026-05-04 18:06:14,Превышение скорости на 20-40 км/ч,75000,86fc3acd3eb134d1fa2e9eef376a2bc5
63282,ca1dfd0174e64b3177ceab21a6eb6274,2026-05-04 18:06:14,Превышение скорости на 20-40 км/ч,75000,338f79e70bb8281012f0832a01283185
13402,ca1dfd0174e64b3177ceab21a6eb6274,2026-05-07 20:32:19,Превышение скорости на 20-40 км/ч,75000,55b74ea917282c4876754b2c542c0e8c
26224,ca1dfd0174e64b3177ceab21a6eb6274,2026-05-07 20:32:19,Превышение скорости на 20-40 км/ч,75000,d9dc731ff6c884d8a51bc4ae00c5bc1c


[fines   ] DUPLICATE_BILL         n=      4  -> оставлено первое постановление, второе помечено флагом


### 2.2 Единица измерения суммы штрафа

Значения — целые, кратные 50 000. Штраф за превышение 20–40 км/ч в 2026 году составляет
750 ₽. Следовательно, сумма указана в **копейках**.

In [12]:
vc = fines.total_fine_amount.value_counts().sort_index()
display(pd.DataFrame({"копейки":vc.index,"рубли":(vc.index/100).astype(int),
                      "штрафов":vc.values}).set_index("копейки"))
fines['fine_rub'] = fines.total_fine_amount/100
print(f"проверка: превышение 20-40 км/ч -> "
      f"{fines.loc[fines.offence_short_statement.str.contains('20-40'),'fine_rub'].mode().iloc[0]:.0f} ₽ "
      "(соответствует действующему тарифу)")
log("fines","AMOUNT_IN_KOPECKS","сумма штрафа указана в копейках, не в рублях",
    len(fines), "добавлена колонка fine_rub = total_fine_amount/100")

,рубли,штрафов
копейки,,
50000,500,395
66666,666,19
75000,750,63566
80000,800,683
100000,1000,2145
106666,1066,30
133333,1333,3
150000,1500,7593
200000,2000,399


проверка: превышение 20-40 км/ч -> 750 ₽ (соответствует действующему тарифу)
[fines   ] AMOUNT_IN_KOPECKS      n= 77,978  -> добавлена колонка fine_rub = total_fine_amount/100


### 2.3 Даты: границы наблюдения

`bill_offence_date` — единственная дата в таблице. Строим суточный ряд и смотрим,
однороден ли он по всему диапазону.

In [13]:
fines['offence_dt']   = fines.bill_offence_date
fines['offence_date'] = fines.offence_dt.dt.normalize()
fines['month']        = fines.offence_dt.dt.to_period('M').astype(str)
print("диапазон:", fines.offence_dt.min(), "..", fines.offence_dt.max())

kept = fines[fines.is_kept_bill]
wk = kept.groupby(kept.offence_dt.dt.to_period('W')).size()
print("\nнедельный ряд постановлений:")
for p,v in wk.items():
    bar = "#"*int(v/80)
    print(f"  {str(p)[:10]}  {v:6,d}  {bar}")

диапазон: 2026-03-20 00:06:15 .. 2026-09-14 15:54:01

недельный ряд постановлений:
  2026-03-16     380  ####
  2026-03-23   1,926  ########################
  2026-03-30   3,073  ######################################
  2026-04-06   2,695  #################################
  2026-04-13   2,802  ###################################
  2026-04-20   2,805  ###################################
  2026-04-27   3,166  #######################################
  2026-05-04   3,085  ######################################
  2026-05-11   3,501  ###########################################
  2026-05-18   4,048  ##################################################
  2026-05-25   3,887  ################################################
  2026-06-01   4,130  ###################################################
  2026-06-08   4,328  ######################################################
  2026-06-15   4,024  ##################################################
  2026-06-22   4,079  ###############################

**Читаем ряд.** Плато ~2 700–4 300 в неделю держится с начала апреля по конец июля.
По краям — провалы: разгон на старте (380 → 1 926 → 3 073) и затухание в конце
(3 176 → … → 597 → 4).

Падение до 4 наблюдений в неделю не может быть поведением: столько нарушений не бывает
у 25 тысяч водителей. Это граница выгрузки. Формализуем два порога.

In [14]:
daily = kept.groupby('offence_date').size().rename('n')
full  = pd.date_range(daily.index.min(), daily.index.max(), freq='D')
daily = daily.reindex(full, fill_value=0)
plateau = daily.loc['2026-04-15':'2026-07-15'].median()
print(f"уровень плато (15.04-15.07): {plateau:.0f} постановлений в день")
print(f"порог 50% от плато        : {0.5*plateau:.0f}\n")

below = daily < 0.5*plateau
left_end  = daily.index[~below][0]
right_beg = daily.index[(daily.index>pd.Timestamp('2026-07-01')) & below][0]
print(f"первый день выше порога            : {left_end.date()}")
print(f"первый день устойчиво ниже порога  : {right_beg.date()}")
print("\nОкругляем до календарных границ месяца, чтобы окно совпадало с месячной панелью:")
MATURE_START, MATURE_END = pd.Timestamp("2026-04-01"), pd.Timestamp("2026-07-31")
print(f"  ЗРЕЛОЕ ОКНО = {MATURE_START.date()} .. {MATURE_END.date()}")

fines['flag_left_truncated'] = fines.offence_date <  MATURE_START
fines['flag_right_censored'] = fines.offence_date >  MATURE_END
fines['in_mature_window']    = (~fines.flag_left_truncated) & (~fines.flag_right_censored)
log("fines","LEFT_TRUNCATION",
    f"суточное число растёт с {daily.iloc[0]:.0f} до плато {plateau:.0f} — выгрузка неполна на старте",
    int(fines.flag_left_truncated.sum()), "помечено; ИСКЛЮЧЕНО из всех расчётов по нарушениям")
log("fines","RIGHT_CENSORING",
    f"с 01.08 монотонное затухание до {daily.iloc[-1]:.0f}/день у даты выгрузки — лаг регистрации",
    int(fines.flag_right_censored.sum()),
    "помечено; ИСКЛЮЧЕНО. Иначе ложный вывод о падении нарушений на ~32%")
print(f"\nв зрелом окне остаётся {int(fines.in_mature_window.sum()):,} из {len(fines):,} постановлений "
      f"({100*fines.in_mature_window.mean():.1f}%)")

уровень плато (15.04-15.07): 524 постановлений в день
порог 50% от плато        : 262

первый день выше порога            : 2026-03-27
первый день устойчиво ниже порога  : 2026-08-24

Округляем до календарных границ месяца, чтобы окно совпадало с месячной панелью:
  ЗРЕЛОЕ ОКНО = 2026-04-01 .. 2026-07-31
[fines   ] LEFT_TRUNCATION        n=  3,032  -> помечено; ИСКЛЮЧЕНО из всех расчётов по нарушениям
[fines   ] RIGHT_CENSORING        n= 12,622  -> помечено; ИСКЛЮЧЕНО. Иначе ложный вывод о падении нарушений на ~32%

в зрелом окне остаётся 62,324 из 77,978 постановлений (79.9%)


### 2.4 Доказательство, что правый край — лаг регистрации, а не падение нарушаемости

Три независимые проверки. Полные графики — в `02_EDA`.

In [15]:
m = kept.groupby(kept.offence_dt.dt.to_period('M')).size()
mat = m[['2026-04','2026-05','2026-06','2026-07']].mean()
print("ПРОВЕРКА 1. Насколько «упали» нарушения, если взять данные как есть:")
print(f"  август {m['2026-08']:,} vs среднее апр-июл {mat:,.0f} = {100*(m['2026-08']/mat-1):.1f}%")
print(f"  последняя неделя выгрузки: {daily.loc['2026-09-08':].sum():.0f} постановлений — физически невозможно\n")

print("ПРОВЕРКА 2. Независимый индикатор активности — заправки не падают:")
_fu = fuel_raw.copy(); _fu['w']=_fu.order_datetime.dt.to_period('W')
wf = _fu.groupby('w').size(); wv = kept.groupby(kept.offence_dt.dt.to_period('W')).size()
cc = pd.DataFrame({'штрафы':wv,'заправки':wf}).dropna()
cc['штрафов_на_1000_заправок'] = (1000*cc.штрафы/cc.заправки).round(1)
display(cc.tail(8))

print("ПРОВЕРКА 3 (решающая). Падение избирательно по типам нарушений:")
jul = kept[kept.month=='2026-07'].offence_short_statement.value_counts()
aug = kept[kept.month=='2026-08'].offence_short_statement.value_counts()
t = pd.DataFrame({'июль':jul,'август':aug}).dropna(); t = t[t.июль>=200]
t['изменение_%'] = (100*(t.август/t.июль-1)).round(1)
display(t.sort_values('изменение_%'))
print("Камерное превышение скорости проседает слабее всех, а типы, требующие ручного")
print("оформления (ремень, телефон, парковка), — вдвое сильнее. Это подпись лага обработки:")
print("реальное падение трафика било бы по всем категориям одинаково.")

ПРОВЕРКА 1. Насколько «упали» нарушения, если взять данные как есть:
  август 10,635 vs среднее апр-июл 15,580 = -31.7%
  последняя неделя выгрузки: 455 постановлений — физически невозможно

ПРОВЕРКА 2. Независимый индикатор активности — заправки не падают:


,штрафы,заправки,штрафов_на_1000_заправок
2026-07-13/2026-07-19,3835,12977.0,295.5
2026-07-20/2026-07-26,3871,12773.0,303.1
2026-07-27/2026-08-02,3176,13762.0,230.8
2026-08-03/2026-08-09,2866,14258.0,201.0
2026-08-10/2026-08-16,2386,14306.0,166.8
2026-08-17/2026-08-23,2220,12902.0,172.1
2026-08-24/2026-08-30,1964,13398.0,146.6
2026-08-31/2026-09-06,1605,8602.0,186.6


ПРОВЕРКА 3 (решающая). Падение избирательно по типам нарушений:


,июль,август,изменение_%
offence_short_statement,,,
Нарушение разметки,638.0,208.0,-67.4
Не пристегнут ремень безопасности,820.0,330.0,-59.8
Использование телефона за рулем,282.0,141.0,-50.0
Превышение скорости на 40-60 км/ч,294.0,193.0,-34.4
Превышение скорости на 20-40 км/ч,13156.0,8931.0,-32.1


Камерное превышение скорости проседает слабее всех, а типы, требующие ручного
оформления (ремень, телефон, парковка), — вдвое сильнее. Это подпись лага обработки:
реальное падение трафика било бы по всем категориям одинаково.


### 2.5 Категории нарушений и целевые переменные гипотезы

In [16]:
print("ВСЕ формулировки нарушений в данных:")
display(kept.offence_short_statement.value_counts().to_frame('постановлений')
        .assign(доля_pct=lambda d:(100*d.постановлений/len(kept)).round(2)))

ВСЕ формулировки нарушений в данных:


,постановлений,доля_pct
offence_short_statement,,
Превышение скорости на 20-40 км/ч,62605,80.29
Не пристегнут ремень безопасности,3718,4.77
Нарушение разметки,3082,3.95
Превышение скорости на 40-60 км/ч,1496,1.92
Использование телефона за рулем,1265,1.62
Проезд на красный сигнал светофора,851,1.09
Движение по выделенной полосе,834,1.07
Пересечение стоп-линии,713,0.91
Остановка или стоянка в неположенном месте (Москва и Санкт-Петербург),491,0.63


In [17]:
# Целевые категории гипотезы -> точные формулировки в данных
TARGET_MAP = {
 "razmetka" : ["Нарушение разметки"],
 "parkovka" : ["Остановка или стоянка в неположенном месте",
               "Остановка или стоянка в неположенном месте (Москва и Санкт-Петербург)"],
 "stop_line": ["Пересечение стоп-линии"],
 "obochina" : ["Движение по обочине"],
 "vydelenka": ["Движение по выделенной полосе",
               "Движение по выделенной полосе (Москва и Санкт-Петербург)"],
 "telefon"  : ["Использование телефона за рулем"],
}
# Укрупнённая категория — для описательной статистики
def categorize(s):
    if "Превышение скорости" in s:
        if "20-40" in s: return "Превышение 20-40"
        if "40-60" in s: return "Превышение 40-60"
        return "Превышение 60+"
    for k,v in [("Разметка / полоса",["Нарушение разметки","Поворот не из крайней полосы"]),
                ("Парковка",["Остановка"]), ("Выделенная полоса",["выделенной полосе"]),
                ("Светофор",["светофора","стоп-линии"]), ("Обочина",["обочине"]),
                ("Телефон за рулём",["телефона"]), ("Ремень безопасности",["ремень"]),
                ("Платная дорога",["платной дороге"]), ("Встречная полоса",["встречного"]),
                ("Запрещённый поворот",["запрещенном месте"]), ("Пешеход",["пешехода"]),
                ("Световые приборы",["световыми"]), ("Грузовые ТС",["грузов","Грузовы"])]:
        if any(x in s for x in v): return k
    return "Прочее"
fines['offence_category'] = fines.offence_short_statement.map(categorize)
fines['is_speeding'] = fines.offence_short_statement.str.contains("Превышение скорости")

stmt2key = {s:k for k,v in TARGET_MAP.items() for s in v}
fines['target_key'] = fines.offence_short_statement.map(stmt2key)
miss_t = [k for k,v in TARGET_MAP.items() if not fines.offence_short_statement.isin(v).any()]
print("целевые категории гипотезы, не найденные в данных:", miss_t if miss_t else "нет — найдены все 6")
display(fines[fines.is_kept_bill & fines.in_mature_window].target_key.value_counts(dropna=False)
        .to_frame('в зрелом окне'))
print("\nВНИМАНИЕ: укрупнённая offence_category НЕПРИГОДНА для проверки гипотезы —")
print("  'Светофор' смешивает красный сигнал и стоп-линию (разные механизмы);")
print("  'Разметка / полоса' смешивает разметку и поворот не из крайней полосы.")
print("  Поэтому все модели работают на уровне offence_short_statement.")

целевые категории гипотезы, не найденные в данных: нет — найдены все 6


,в зрелом окне
target_key,
NaN,56235
razmetka,2574
telefon,1018
vydelenka,850
parkovka,706
stop_line,564
obochina,373



ВНИМАНИЕ: укрупнённая offence_category НЕПРИГОДНА для проверки гипотезы —
  'Светофор' смешивает красный сигнал и стоп-линию (разные механизмы);
  'Разметка / полоса' смешивает разметку и поворот не из крайней полосы.
  Поэтому все модели работают на уровне offence_short_statement.


In [18]:
fines.to_csv(OUT/"clean_03_fines.csv", sep=';', index=False, encoding='utf-8-sig')
print(f"clean_03_fines.csv: {len(fines):,} строк, {fines.shape[1]} колонок (все строки сохранены, решения — во флагах)")

clean_03_fines.csv: 77,978 строк, 19 колонок (все строки сохранены, решения — во флагах)


---
## 3. Таблица топливных транзакций

### 3.1 Отрицательные объёмы — возвраты

In [19]:
fuel = fuel_raw.copy()
fuel['order_dt']   = fuel.order_datetime
fuel['order_date'] = fuel.order_dt.dt.normalize()
fuel['month']      = fuel.order_dt.dt.to_period('M').astype(str)
neg = fuel.order_fuel_volume<=0
print(f"объём <= 0: {int(neg.sum()):,} транзакций")
print(f"  средний отрицательный объём : {fuel.loc[neg,'order_fuel_volume'].mean():.2f} л")
print(f"  средний положительный объём : {fuel.loc[~neg,'order_fuel_volume'].mean():.2f} л")
print("  зеркальность средних -> это возвраты/отмены, а не ошибки измерения")
display(fuel[neg].groupby('month').size().to_frame('возвратов')
        .join(fuel.groupby('month').size().to_frame('всего'))
        .assign(доля_pct=lambda d:(100*d.возвратов/d.всего).round(2)))
fuel['flag_reversal'] = neg
log("fuel","REVERSAL",
    f"объём <= 0 (минимум {fuel.order_fuel_volume.min():.1f} л); среднее зеркально положительному; "
    "равномерно по месяцам",
    int(neg.sum()), "помечено; ИСКЛЮЧЕНО из агрегатов объёма и цены, строка сохранена")

объём <= 0: 1,500 транзакций
  средний отрицательный объём : -36.16 л
  средний положительный объём : 36.26 л
  зеркальность средних -> это возвраты/отмены, а не ошибки измерения


,возвратов,всего,доля_pct
month,,,
2026-03,104,25061,0.41
2026-04,305,75104,0.41
2026-05,305,80865,0.38
2026-06,290,68511,0.42
2026-07,230,62492,0.37
2026-08,239,60913,0.39
2026-09,27,6753,0.40


[fuel    ] REVERSAL               n=  1,500  -> помечено; ИСКЛЮЧЕНО из агрегатов объёма и цены, строка сохранена


### 3.2 Цена: три режима вместо одного

Гистограмма цены за литр трёхмодальна. Разделяем по тарифным группам — иначе газ и
не-моторные позиции исказят все средние и сделают «газовые» регионы ложно освобождёнными
от лимита.

In [20]:
p, v = fuel.order_fuel_price_1liter, fuel.order_fuel_volume
tiers = [(0,50,"СУГ/КПГ (газ)"),(50,110,"Бензин/ДТ"),(110,10**6,"не моторное топливо")]
rows=[]
for lo,hi,nm in tiers:
    m = p.between(lo,hi,inclusive='left')
    rows.append({"тарифная группа":nm,"диапазон ₽/л":f"{lo}-{hi if hi<10**6 else '∞'}",
                 "транзакций":int(m.sum()),"доля_%":round(100*m.mean(),2),
                 "медиана цены":round(p[m].median(),2),"медиана объёма":round(v[m].median(),2),
                 "p90 объёма":round(v[m].quantile(.9),2)})
display(pd.DataFrame(rows))
def tier(x):
    if x<50: return "СУГ/КПГ (газ)"
    if x<110: return "Бензин/ДТ"
    return "не моторное топливо"
fuel['product_tier'] = p.map(tier)
fuel['in_analysis']  = (fuel.product_tier=="Бензин/ДТ") & (~fuel.flag_reversal)
log("fuel","PRICE_MULTIMODAL",
    "распределение цены трёхмодальное: газ ~31 ₽/л, бензин/ДТ 50-110 ₽/л (97.3%), "
    "не моторное топливо ~259 ₽/л при объёме 6.6 л",
    int((fuel.product_tier!="Бензин/ДТ").sum()),
    "разделено по тарифным группам; основной анализ — только Бензин/ДТ")

big = v>100
fuel['flag_large_volume'] = big
log("fuel","LARGE_VOLUME",
    f"объём > 100 л (максимум {v.max():.1f} л) — возможно коммерческие/бензовозные заправки",
    int(big.sum()), "СОХРАНЕНО; все ключевые метрики считаются по перцентилям, устойчивым к хвосту")

,тарифная группа,диапазон ₽/л,транзакций,доля_%,медиана цены,медиана объёма,p90 объёма
0,СУГ/КПГ (газ),0-50,7122,1.88,30.92,45.62,78.44
1,Бензин/ДТ,50-110,369301,97.26,69.09,35.62,53.62
2,не моторное топливо,110-∞,3276,0.86,259.00,6.62,25.62


[fuel    ] PRICE_MULTIMODAL       n= 10,398  -> разделено по тарифным группам; основной анализ — только Бензин/ДТ
[fuel    ] LARGE_VOLUME           n=  1,060  -> СОХРАНЕНО; все ключевые метрики считаются по перцентилям, устойчивым к хвосту


### 3.3 Квантование объёма — подпись режима нормирования

Неожиданная находка: объёмы сильно сконцентрированы на значениях вида `X.62`.

In [21]:
frac = (v - np.floor(v)).round(2)
print(f"доля транзакций с дробной частью .62 : {100*(frac==0.62).mean():.1f}%")
display(v.value_counts().head(8).to_frame('транзакций')
        .assign(доля_pct=lambda d:(100*d.транзакций/len(fuel)).round(2)))
print("Самые частые объёмы — 35.62, 45.62, 25.62, 55.62 л. Это не случайность:")
print("именно эти значения окажутся потолками заправки в фазах кризиса (раздел 3.4).")
print("Водители упираются в лимит и заливают ровно «под потолок».")
log("fuel","VOLUME_QUANTIZATION",
    "36.6% объёмов имеют дробную часть .62; модальные значения 35.62/45.62/55.62 л "
    "совпадают с лимитами заправки",
    int((frac==0.62).sum()), "сохранено; используется как индикатор связанности лимитом")

доля транзакций с дробной частью .62 : 36.6%


,транзакций,доля_pct
order_fuel_volume,,
35.62,40954,10.79
45.62,20397,5.37
25.62,13539,3.57
55.62,7854,2.07
30.62,7731,2.04
40.62,6974,1.84
20.62,5367,1.41
15.62,5204,1.37


Самые частые объёмы — 35.62, 45.62, 25.62, 55.62 л. Это не случайность:
именно эти значения окажутся потолками заправки в фазах кризиса (раздел 3.4).
Водители упираются в лимит и заливают ровно «под потолок».
[fuel    ] VOLUME_QUANTIZATION    n=138,997  -> сохранено; используется как индикатор связанности лимитом


### 3.4 Определение фаз кризиса **из данных**

Дата начала кризиса не берётся из внешнего источника. Она выводится из суточного
90-го перцентиля объёма заправки и доли заправок свыше 50 л.

In [22]:
G = fuel[fuel.in_analysis].copy()
d90 = G.groupby('order_date').order_fuel_volume.agg(
        p90=lambda s:s.quantile(.9), p99=lambda s:s.quantile(.99),
        share_gt50=lambda s:100*(s>50).mean(), n='size')
print("суточный p90 и доля заправок >50 л, 15.06 - 08.07:")
display(d90.loc['2026-06-15':'2026-07-08'].round(2))

суточный p90 и доля заправок >50 л, 15.06 - 08.07:


,p90,p99,share_gt50,n
order_date,,,,
2026-06-15,56.89,90.45,21.31,2098
2026-06-16,56.52,77.44,22.59,2081
2026-06-17,55.62,75.77,19.64,2144
2026-06-18,56.98,109.83,19.56,2720
2026-06-19,54.21,80.62,15.63,2547
2026-06-20,54.62,80.74,16.08,2065
2026-06-21,53.72,85.77,14.64,2349
2026-06-22,53.62,85.62,14.76,2127
2026-06-23,50.62,70.62,10.89,2020


In [23]:
PHASES = [("P1_база",           "2026-03-20","2026-06-18", None),
          ("P2_ужесточение",    "2026-06-19","2026-06-23", None),
          ("P3_лимит_45.62л",   "2026-06-24","2026-07-03", 45.62),
          ("P4_лимит_35.62л",   "2026-07-04","2026-07-27", 35.62),
          ("P5_частич_ослабл",  "2026-07-28","2026-08-16", 55.62),
          ("P6_лимит_45.62л_2", "2026-08-17","2026-09-04", 45.62)]
rows=[]
for nm,a,b,lim in PHASES:
    s = G[(G.order_date>=a)&(G.order_date<=b)]
    rows.append({"фаза":nm,"с":a,"по":b,"лимит_л":lim,"транзакций":len(s),
                 "p90":round(s.order_fuel_volume.quantile(.9),2),
                 "p99":round(s.order_fuel_volume.quantile(.99),2),
                 "доля>50л_%":round(100*(s.order_fuel_volume>50).mean(),2),
                 "медиана ₽/л":round(s.order_fuel_price_1liter.median(),2),
                 "p95 ₽/л":round(s.order_fuel_price_1liter.quantile(.95),2)})
PH = pd.DataFrame(rows); display(PH)

def phase_of(dt):
    for nm,a,b,_ in PHASES:
        if pd.Timestamp(a) <= dt <= pd.Timestamp(b): return nm
    return "вне окна"
fuel['crisis_phase'] = fuel.order_date.map(phase_of)
fines['crisis_phase'] = fines.offence_date.map(phase_of)

CRISIS_ONSET = pd.Timestamp("2026-06-19")   # первый день устойчивого снижения p90
HARD_LIMIT   = pd.Timestamp("2026-06-24")   # обвал доли заправок >50 л: 10.9% -> 2.2%
print("\nВЫВОД ПО ДАТИРОВКЕ КРИЗИСА (получен из данных, без внешних источников):")
print("  до 18.06 включительно  p90 держится на 55-57 л, доля заправок >50 л ~20%")
print("  19.06 - 23.06          p90 сползает 54.2 -> 50.6, доля >50 л падает 15.6% -> 10.9%")
print("  24.06                  ОБВАЛ: доля заправок >50 л 10.9% -> 2.2%, p90 фиксируется на 45.62")
print("  04.07                  второе ужесточение: p90 -> 35.62")
print("  28.07 - 16.08          частичное ослабление, но потолок 55.62 л сохраняется (p99 = 55.62)")
print("  17.08                  возврат лимита 45.62 л")
print(f"\n  НАЧАЛО КРИЗИСА = {CRISIS_ONSET.date()}, ЖЁСТКИЙ ЛИМИТ = {HARD_LIMIT.date()}")
print("  Шок количественный: объём -37%, медианная цена всего +3%, но p95 цены +22%.")
print("  МАЙ ЦЕЛИКОМ ЛЕЖИТ В ДОКРИЗИСНОЙ БАЗЕ P1.")
log("fuel","CRISIS_PHASES_DERIVED",
    "фазы режима нормирования выведены из суточного p90 объёма и доли заправок >50 л",
    len(fuel), f"начало кризиса {CRISIS_ONSET.date()}, жёсткий лимит {HARD_LIMIT.date()}")

,фаза,с,по,лимит_л,транзакций,p90,p99,доля>50л_%,медиана ₽/л,p95 ₽/л
0,P1_база,2026-03-20,2026-06-18,NaN,218139,55.85,79.11,20.68,68.49,79.29
1,P2_ужесточение,2026-06-19,2026-06-23,NaN,11108,53.52,80.79,14.48,69.24,91.18
2,P3_лимит_45.62л,2026-06-24,2026-07-03,45.62,19247,45.62,55.62,1.60,69.45,92.64
3,P4_лимит_35.62л,2026-07-04,2026-07-27,35.62,46898,35.62,45.62,0.73,70.20,85.99
4,P5_частич_ослабл,2026-07-28,2026-08-16,55.62,39157,51.21,55.62,12.08,70.51,94.29
5,P6_лимит_45.62л_2,2026-08-17,2026-09-04,45.62,33288,45.62,55.67,3.14,70.52,96.90



ВЫВОД ПО ДАТИРОВКЕ КРИЗИСА (получен из данных, без внешних источников):
  до 18.06 включительно  p90 держится на 55-57 л, доля заправок >50 л ~20%
  19.06 - 23.06          p90 сползает 54.2 -> 50.6, доля >50 л падает 15.6% -> 10.9%
  24.06                  ОБВАЛ: доля заправок >50 л 10.9% -> 2.2%, p90 фиксируется на 45.62
  04.07                  второе ужесточение: p90 -> 35.62
  28.07 - 16.08          частичное ослабление, но потолок 55.62 л сохраняется (p99 = 55.62)
  17.08                  возврат лимита 45.62 л

  НАЧАЛО КРИЗИСА = 2026-06-19, ЖЁСТКИЙ ЛИМИТ = 2026-06-24
  Шок количественный: объём -37%, медианная цена всего +3%, но p95 цены +22%.
  МАЙ ЦЕЛИКОМ ЛЕЖИТ В ДОКРИЗИСНОЙ БАЗЕ P1.
[fuel    ] CRISIS_PHASES_DERIVED  n=379,699  -> начало кризиса 2026-06-19, жёсткий лимит 2026-06-24


In [24]:
fuel['spend_rub'] = fuel.order_fuel_volume*fuel.order_fuel_price_1liter
fuel.to_csv(OUT/"clean_04_fuel.csv", sep=';', index=False, encoding='utf-8-sig')
PH.to_csv(OUT/"crisis_phases.csv", sep=';', index=False, encoding='utf-8-sig')
print(f"clean_04_fuel.csv: {len(fuel):,} строк")

clean_04_fuel.csv: 379,699 строк


---
## 4. Фиксированная когорта

Знаменатель для любых ставок — **фиксированная** когорта клиентов, подключившихся
до начала зрелого окна. Использовать «активных клиентов месяца» нельзя: их число падает
вместе с кризисом, и деление на него уничтожит сам эффект, который мы ищем.

In [25]:
FIXED = set(clients_lv.loc[clients_lv.subscription_dt < MATURE_START, 'client_id'])
print(f"всего клиентов                     : {len(clients_lv):,}")
print(f"подключились до {MATURE_START.date()}      : {len(FIXED):,}  <- ФИКСИРОВАННАЯ КОГОРТА")
print(f"подключились внутри/после окна     : {len(clients_lv)-len(FIXED):,}  (исключены)")

act = fuel[fuel.in_analysis].groupby('month').client_id.nunique()
print("\nПочему нельзя делить на «активных клиентов месяца»:")
display(act.to_frame('активных клиентов').assign(
    к_апрелю_pct=lambda d:(100*d['активных клиентов']/d['активных клиентов'].iloc[1]-100).round(1)))
print("Число активных падает на ~18% — это и есть эффект кризиса, а не смена базы.")

всего клиентов                     : 25,675
подключились до 2026-04-01      : 23,452  <- ФИКСИРОВАННАЯ КОГОРТА
подключились внутри/после окна     : 2,223  (исключены)

Почему нельзя делить на «активных клиентов месяца»:


,активных клиентов,к_апрелю_pct
month,,
2026-03,13745,-28.7
2026-04,19289,0.0
2026-05,19869,3.0
2026-06,18777,-2.7
2026-07,16120,-16.4
2026-08,16337,-15.3
2026-09,4912,-74.5


Число активных падает на ~18% — это и есть эффект кризиса, а не смена базы.


---
## 5. Панель клиент × месяц

Основная аналитическая таблица. Прямоугольник: каждый клиент фиксированной когорты
присутствует в каждом месяце зрелого окна. **Ноль — это реальный ноль**, а не пропуск.

In [26]:
MONTHS = ['2026-04','2026-05','2026-06','2026-07']
base = pd.MultiIndex.from_product([sorted(FIXED), MONTHS], names=['client_id','month'])

V = fines[fines.is_kept_bill & fines.in_mature_window & fines.client_id.isin(FIXED)]
F = fuel[fuel.in_analysis & fuel.client_id.isin(FIXED) & fuel.month.isin(MONTHS)]

fa = F.groupby(['client_id','month']).agg(
        litres=('order_fuel_volume','sum'), fuel_spend_rub=('spend_rub','sum'),
        n_fuel_tx=('order_id','size'), vol_mean=('order_fuel_volume','mean'),
        vol_max=('order_fuel_volume','max'), price_mean=('order_fuel_price_1liter','mean'))
va = V.groupby(['client_id','month']).agg(
        n_violations=('bill_id','size'), fine_sum_rub=('fine_rub','sum'),
        n_speeding=('is_speeding','sum'), n_offence_categories=('offence_category','nunique'))
tg = V.pivot_table(index=['client_id','month'], columns='target_key', aggfunc='size', fill_value=0)

panel = pd.DataFrame(index=base).join([fa, va, tg]).reset_index()
for c in ['litres','fuel_spend_rub','n_fuel_tx','n_violations','fine_sum_rub',
          'n_speeding','n_offence_categories']+list(TARGET_MAP):
    if c in panel: panel[c] = panel[c].fillna(0)
    else: panel[c] = 0
COMPOSITE_6 = list(TARGET_MAP)                                  # все 6 категорий гипотезы
COMPOSITE_5 = [k for k in TARGET_MAP if k!='telefon']           # только манёвренные
panel['crisis_related_violations'] = panel[COMPOSITE_6].sum(axis=1)
panel['manoeuvre_violations']      = panel[COMPOSITE_5].sum(axis=1)
panel['month_idx'] = panel.month.map({m:i for i,m in enumerate(MONTHS)})
panel['post_crisis'] = (panel.month_idx>=3).astype(int)   # июль = первый полностью кризисный месяц

meta_cols = ['client_id','n_vehicles','gender','age_type_code','kladr_code','region_name',
             'auto_mark','auto_year','vehicle_age_2026','price','engine_litres','engine_hp',
             'subscription_dt','f2025_apr_aug','april_2025_fines','may_2025_fines',
             'jun_2025_fines','jul_2025_fines','aug_2025_fines']
panel = panel.merge(clients_lv[meta_cols], on='client_id', how='left')
print(f"панель: {len(panel):,} строк = {panel.client_id.nunique():,} клиентов x {panel.month.nunique()} месяца")

панель: 93,808 строк = 23,452 клиентов x 4 месяца


### 5.1 Валидация панели

In [27]:
ok = True
def v(name, cond, detail):
    global ok; ok &= bool(cond)
    print(("  OK   " if cond else " !!!  ")+name+" | "+str(detail))

v("прямоугольник client x month", len(panel)==panel.client_id.nunique()*len(MONTHS),
  f"{panel.client_id.nunique():,} x {len(MONTHS)} = {len(panel):,}")
v("нет дублей (client, month)", not panel.duplicated(['client_id','month']).any(),
  f"дублей {panel.duplicated(['client_id','month']).sum()}")
v("клиенты == фиксированная когорта", set(panel.client_id)==FIXED, f"{len(FIXED):,}")
v("сумма n_violations == исходные штрафы", panel.n_violations.sum()==len(V),
  f"{int(panel.n_violations.sum()):,} vs {len(V):,}")
v("сумма литров == исходные транзакции",
  abs(panel.litres.sum()-F.order_fuel_volume.sum())<1,
  f"{panel.litres.sum():,.1f} vs {F.order_fuel_volume.sum():,.1f}")
for k in TARGET_MAP:
    v(f"  сверка категории {k}", panel[k].sum()==int((V.target_key==k).sum()),
      f"{int(panel[k].sum()):,} vs {int((V.target_key==k).sum()):,}")
v("нет отрицательных значений", (panel[['litres','n_violations']]>=0).all().all(), "ок")
print("\nВСЕ ПРОВЕРКИ ПРОЙДЕНЫ" if ok else "\nЕСТЬ РАСХОЖДЕНИЯ — СМОТРЕТЬ ВЫШЕ")

print(f"\nдоля нулевых клиенто-месяцев по n_violations: {100*(panel.n_violations==0).mean():.1f}%")
print(f"среднее {panel.n_violations.mean():.3f}, дисперсия {panel.n_violations.var():.3f}, "
      f"var/mean = {panel.n_violations.var()/panel.n_violations.mean():.2f} -> сверхдисперсия")
display(panel.groupby('month')[['n_violations','crisis_related_violations',
                                'manoeuvre_violations','litres','n_fuel_tx']].sum().round(0))

  OK   прямоугольник client x month | 23,452 x 4 = 93,808
  OK   нет дублей (client, month) | дублей 0
  OK   клиенты == фиксированная когорта | 23,452
  OK   сумма n_violations == исходные штрафы | 58,566 vs 58,566
  OK   сумма литров == исходные транзакции | 9,288,950.8 vs 9,288,950.8
  OK     сверка категории razmetka | 2,414 vs 2,414
  OK     сверка категории parkovka | 652 vs 652
  OK     сверка категории stop_line | 526 vs 526
  OK     сверка категории obochina | 353 vs 353
  OK     сверка категории vydelenka | 806 vs 806
  OK     сверка категории telefon | 963 vs 963
  OK   нет отрицательных значений | ок

ВСЕ ПРОВЕРКИ ПРОЙДЕНЫ

доля нулевых клиенто-месяцев по n_violations: 72.0%
среднее 0.624, дисперсия 2.489, var/mean = 3.99 -> сверхдисперсия


,n_violations,crisis_related_violations,manoeuvre_violations,litres,n_fuel_tx
month,,,,,
2026-04,11554.0,1056.0,896.0,2567767.0,67072.0
2026-05,15442.0,1470.0,1256.0,2752147.0,72189.0
2026-06,16401.0,1804.0,1473.0,2261917.0,61034.0
2026-07,15169.0,1384.0,1126.0,1707120.0,54906.0


In [28]:
panel.to_csv(OUT/"clean_05_panel_client_month.csv", sep=';', index=False, encoding='utf-8-sig')
print(f"clean_05_panel_client_month.csv: {len(panel):,} строк x {panel.shape[1]} колонок")

clean_05_panel_client_month.csv: 93,808 строк x 40 колонок


---
## 6. Журнал очистки

Полный список решений. Ни одна строка не удалена из файлов — всё помечено флагами,
фильтрация происходит на этапе анализа.

In [29]:
LOGDF = pd.DataFrame(LOG)
display(LOGDF)
LOGDF.to_csv(OUT/"cleaning_log.csv", sep=';', index=False, encoding='utf-8-sig')

META = {
 "сырые_файлы":{"clients_demographics.csv":len(clients_raw),"fines_2026.csv":len(fines_raw),
                "fuel_transaction.csv":len(fuel_raw),"итого_строк":TOTAL_ROWS,"итого_признаков":TOTAL_COLS},
 "очищенные_файлы":{"clean_01_vehicles.csv":len(vehicles),"clean_02_clients.csv":len(clients_lv),
                    "clean_03_fines.csv":len(fines),"clean_04_fuel.csv":len(fuel),
                    "clean_05_panel_client_month.csv":len(panel)},
 "окна":{"зрелое_окно_нарушений":f"{MATURE_START.date()} .. {MATURE_END.date()}",
         "топливо":f"{fuel.order_date.min().date()} .. {fuel.order_date.max().date()}"},
 "кризис":{"начало_плавное":str(CRISIS_ONSET.date()),"жёсткий_лимит":str(HARD_LIMIT.date()),
           "фазы":[{"фаза":n,"с":a,"по":b,"лимит_л":l} for n,a,b,l in PHASES]},
 "когорта":{"фиксированная_n":len(FIXED)},
 "целевые_категории":TARGET_MAP,
 "запрещённые_переменные":FORBIDDEN,
}
(OUT/"metadata_derived.json").write_text(json.dumps(META, ensure_ascii=False, indent=2), encoding='utf-8')
print("\nФАЙЛЫ НА ВЫХОДЕ:")
for f in sorted(OUT.glob("*")): print(f"  {f.name:38s} {f.stat().st_size/1024:8.1f} KB")
print(f"\nГотово. Вход: {TOTAL_ROWS:,} сырых строк -> выход: панель {len(panel):,} клиенто-месяцев.")

,таблица,код,что_обнаружено,строк,решение
0,clients,DUP_VEHICLE_ROW,"пара (client_id, auto_document_id) встречается...",47,оставлена строка с максимальным fines_last_12_...
1,clients,VEHICLE_MULTI_CLIENT,один auto_document_id связан с несколькими cli...,196,сохранено как есть — это не ошибка данных
2,clients,NULL_GENDER,пропуск в поле gender,42,"НЕ импутировано; строка остаётся, выпадает тол..."
3,clients,NULL_AGE_TYPE_CODE,пропуск в поле age_type_code,2,"НЕ импутировано; строка остаётся, выпадает тол..."
4,clients,NULL_ENGINE_TYPE,пропуск в поле engine_type,2,"НЕ импутировано; строка остаётся, выпадает тол..."
5,clients,NULL_PRICE,пропуск в поле price,631,"НЕ импутировано; строка остаётся, выпадает тол..."
6,clients,NULL_COLOR,пропуск в поле color,1408,"НЕ импутировано; строка остаётся, выпадает тол..."
7,clients,SUBSCRIBED_MID_WINDOW,клиент подключился после 01.04.2026 — у него н...,2650,помечен; исключён из фиксированной когорты для...
8,clients,NO_2025_BASELINE,клиент подключился после 01.04.2025 — счётчики...,11727,помечен; исключён из всех межгодовых сравнений...
9,clients,POST_TREATMENT_WINDOW,fines_last_6/12/24/36 пересекаются с периодом ...,28237,ЗАПРЕЩЕНЫ как контрольные переменные; вместо н...



ФАЙЛЫ НА ВЫХОДЕ:
  clean_01_vehicles.csv                    6194.3 KB
  clean_02_clients.csv                     3990.0 KB
  clean_03_fines.csv                      24559.5 KB
  clean_04_fuel.csv                       78023.1 KB
  clean_05_panel_client_month.csv         21587.3 KB
  cleaning_log.csv                            5.0 KB
  crisis_phases.csv                           0.6 KB
  eda_daily_series.csv                        4.7 KB
  eda_findings.csv                            2.5 KB
  eda_required_indicators.csv                 2.0 KB
  metadata_derived.json                       2.6 KB
  results_did.csv                             2.8 KB
  results_heterogeneity.csv                   0.6 KB
  results_offset.csv                          2.4 KB
  results_panel_fe.csv                        3.7 KB
  results_placebo_rolling.csv                10.8 KB
  results_robustness.csv                      4.1 KB

Готово. Вход: 485,914 сырых строк -> выход: панель 93,808 клиенто-месяцев.


---
# ЧАСТЬ 2. Что из гипотезы мы вообще можем измерить

Прежде чем строить модели — честная ревизия: какое звено причинной цепочки
наблюдается напрямую, какое только через proxy, а какое не наблюдается вовсе.

In [30]:
import statsmodels.api as sm
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.stattools import durbin_watson
from scipy import stats, optimize
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, matplotlib.dates as mdates
plt.rcParams.update({"figure.dpi":120,"savefig.dpi":150,"font.size":10,"axes.grid":True,
                     "grid.alpha":.25,"axes.spines.top":False,"axes.spines.right":False,
                     "figure.autolayout":True})
GREEN,RED,NAVY,GREY,ORANGE = "#1f5c3a","#c0392b","#1f3a6e","#8a8a8a","#b3541e"
FINAL = RAW/"clean dataset + visuals"/"final_output"; FINAL.mkdir(parents=True, exist_ok=True)
FIG   = FINAL/"figures"; FIG.mkdir(exist_ok=True)

CHAIN = pd.DataFrame([
 ["1. Дефицит топлива / ограничение продаж",
  "падение доступного объёма, появление потолка заправки",
  "379 699 транзакций: объём, цена, дата, клиент",
  "DIRECT",
  "суточный p90 объёма, доля заправок >50 л, медиана и p95 цены",
  "ИЗМЕРЕНО: p90 56→45.62→35.62 л, доля >50 л 20.9%→0.7%"],
 ["2. Очереди на АЗС",
  "рост длины очереди и времени ожидания",
  "переменной очереди в данных НЕТ",
  "NOT OBSERVED",
  "не проверяется напрямую; используются proxy звена 1",
  "НЕ ИЗМЕРЕНО — принципиальный разрыв цепочки"],
 ["3. Изменение поведения водителей",
  "иные маршруты, иная частота и время поездок",
  "частота заправок, объём на клиента, число активных клиентов",
  "PROXY",
  "динамика n_fuel_tx и литров на клиента фиксированной когорты",
  "ИЗМЕРЕНО КОСВЕННО: активных клиентов −18%"],
 ["4. Целевые нарушения",
  "рост 6 категорий, сопутствующих заправке",
  "77 978 постановлений со статьёй и датой",
  "DIRECT",
  "суточные счётчики 6 категорий + composite",
  "ИЗМЕРЕНО: это и есть зависимая переменная"],
], columns=["Звено гипотезы","Что ожидалось","Что есть в данных",
            "Direct / Proxy / Not observed","Как проверяется","Результат"])
display(CHAIN)
CHAIN.to_csv(FINAL/"chain_measurability.csv", sep=';', index=False, encoding='utf-8-sig')
print("КЛЮЧЕВОЕ ОГРАНИЧЕНИЕ, зафиксированное до всех моделей:")
print("  звено 2 (очереди) в данных отсутствует. Топливные транзакции — это НЕ измерение")
print("  очереди, а измерение режима продажи топлива. Поэтому проверяется не вся цепочка,")
print("  а импликация: 'там и тогда, где ограничение действовало, целевых нарушений больше'.")

,Звено гипотезы,Что ожидалось,Что есть в данных,Direct / Proxy / Not observed,Как проверяется,Результат
0,1. Дефицит топлива / ограничение продаж,"падение доступного объёма, появление потолка з...","379 699 транзакций: объём, цена, дата, клиент",DIRECT,"суточный p90 объёма, доля заправок >50 л, меди...","ИЗМЕРЕНО: p90 56→45.62→35.62 л, доля >50 л 20...."
1,2. Очереди на АЗС,рост длины очереди и времени ожидания,переменной очереди в данных НЕТ,NOT OBSERVED,не проверяется напрямую; используются proxy зв...,НЕ ИЗМЕРЕНО — принципиальный разрыв цепочки
2,3. Изменение поведения водителей,"иные маршруты, иная частота и время поездок","частота заправок, объём на клиента, число акти...",PROXY,динамика n_fuel_tx и литров на клиента фиксиро...,ИЗМЕРЕНО КОСВЕННО: активных клиентов −18%
3,4. Целевые нарушения,"рост 6 категорий, сопутствующих заправке",77 978 постановлений со статьёй и датой,DIRECT,суточные счётчики 6 категорий + composite,ИЗМЕРЕНО: это и есть зависимая переменная


КЛЮЧЕВОЕ ОГРАНИЧЕНИЕ, зафиксированное до всех моделей:
  звено 2 (очереди) в данных отсутствует. Топливные транзакции — это НЕ измерение
  очереди, а измерение режима продажи топлива. Поэтому проверяется не вся цепочка,
  а импликация: 'там и тогда, где ограничение действовало, целевых нарушений больше'.


---
# ЧАСТЬ 3. FUEL ANALYSIS — что произошло с топливом и когда

Май зафиксирован как начало кризиса условием проекта. Отдельно устанавливаем,
**когда внутри кризиса появилось количественное ограничение на разовую заправку** —
это наблюдаемое событие, которое можно использовать как вторую спецификацию.

In [31]:
F_ALL = fuel[fuel.in_analysis & fuel.client_id.isin(FIXED)].copy()
fd = F_ALL.groupby('order_date').agg(
        litres=('order_fuel_volume','sum'), n=('order_id','size'),
        p90=('order_fuel_volume', lambda s: s.quantile(.9)),
        p99=('order_fuel_volume', lambda s: s.quantile(.99)),
        share50=('order_fuel_volume', lambda s: 100*(s>50).mean()),
        med_price=('order_fuel_price_1liter','median'),
        p95_price=('order_fuel_price_1liter', lambda s: s.quantile(.95))).reset_index()

MAY_START = pd.Timestamp("2026-05-01")     # НАЧАЛО КРИЗИСА (условие проекта)
LIMIT_DT  = pd.Timestamp("2026-06-19")     # введение лимита на разовую заправку

b = F_ALL[F_ALL.crisis_phase=="P1_база"]; c4 = F_ALL[F_ALL.crisis_phase=="P4_лимит_35.62л"]
apr_ = fd[(fd.order_date>=MATURE_START)&(fd.order_date<MAY_START)]
may_ = fd[(fd.order_date>=MAY_START)&(fd.order_date<pd.Timestamp("2026-06-01"))]
print("ЧТО ПРОИЗОШЛО С ТОПЛИВОМ")
print(f"  объём/день   : {b.groupby('order_date').order_fuel_volume.sum().mean()/1000:.1f} -> "
      f"{c4.groupby('order_date').order_fuel_volume.sum().mean()/1000:.1f} тыс. л "
      f"({100*(c4.groupby('order_date').order_fuel_volume.sum().mean()/b.groupby('order_date').order_fuel_volume.sum().mean()-1):+.1f}%)")
print(f"  доля >50 л   : {100*(b.order_fuel_volume>50).mean():.2f}% -> {100*(c4.order_fuel_volume>50).mean():.2f}%")
print(f"  медиана цены : {b.order_fuel_price_1liter.median():.2f} -> {c4.order_fuel_price_1liter.median():.2f} ₽/л "
      f"({100*(c4.order_fuel_price_1liter.median()/b.order_fuel_price_1liter.median()-1):+.1f}%)")
print("  ТИП ШОКА: количественный (нормирование), а не ценовой\n")
print("КОГДА ПОЯВИЛОСЬ КОЛИЧЕСТВЕННОЕ ОГРАНИЧЕНИЕ")
print(f"  апрель p90 {apr_.p90.median():.2f} л -> май p90 {may_.p90.median():.2f} л "
      f"({100*(may_.p90.median()/apr_.p90.median()-1):+.1f}%)")
print("  -> в мае кризис уже идёт, но ЛИМИТА НА ОБЪЁМ ЗАПРАВКИ ещё нет")
display(fd[(fd.order_date>=pd.Timestamp('2026-06-15'))&(fd.order_date<=pd.Timestamp('2026-06-26'))]
        [['order_date','p90','share50','n']].round(2).set_index('order_date'))
print(f"  первое ужесточение {LIMIT_DT.date()}, жёсткий лимит 45.62 л с 24.06, 35.62 л с 04.07")

ЧТО ПРОИЗОШЛО С ТОПЛИВОМ
  объём/день   : 84.1 -> 54.2 тыс. л (-35.5%)
  доля >50 л   : 20.92% -> 0.71%
  медиана цены : 68.50 -> 70.20 ₽/л (+2.5%)
  ТИП ШОКА: количественный (нормирование), а не ценовой

КОГДА ПОЯВИЛОСЬ КОЛИЧЕСТВЕННОЕ ОГРАНИЧЕНИЕ
  апрель p90 56.11 л -> май p90 56.09 л (-0.0%)
  -> в мае кризис уже идёт, но ЛИМИТА НА ОБЪЁМ ЗАПРАВКИ ещё нет


,p90,share50,n
order_date,,,
2026-06-15,56.89,21.23,1908
2026-06-16,56.84,22.68,1922
2026-06-17,55.62,19.77,1978
2026-06-18,57.69,19.91,2491
2026-06-19,54.54,16.03,2346
2026-06-20,54.79,16.23,1892
2026-06-21,53.44,14.48,2154
2026-06-22,53.63,14.92,1924
2026-06-23,50.62,10.80,1851


  первое ужесточение 2026-06-19, жёсткий лимит 45.62 л с 24.06, 35.62 л с 04.07


---
# ЧАСТЬ 4. TARGET VIOLATIONS — построение зависимой переменной

Зависимая переменная строится из `fines_2026.csv` на уровне **точной формулировки статьи**
(`offence_short_statement`), а не укрупнённой категории: в данных «Светофор» смешивает
проезд на красный и пересечение стоп-линии, а «Разметка / полоса» — разметку и поворот
не из крайней полосы. Для проверки гипотезы такое смешение недопустимо.

In [32]:
NEGCTRL = {"speed_20_40":["Превышение скорости на 20-40 км/ч"],
           "remen":["Не пристегнут ремень безопасности"],
           "platnaya":["Неоплаченный проезд по платной дороге"],
           "svet":["Нарушение правил пользования световыми приборами, звуковыми сигналами"]}
ALLOUT = {**TARGET_MAP, **NEGCTRL}
RU = {"razmetka":"Нарушение разметки","parkovka":"Остановка/стоянка в неположенном месте",
      "stop_line":"Пересечение стоп-линии","obochina":"Движение по обочине",
      "vydelenka":"Движение по выделенной полосе","telefon":"Телефон за рулём",
      "speed_20_40":"[K] Превышение 20-40 км/ч","remen":"[K] Ремень безопасности",
      "platnaya":"[K] Платная дорога","svet":"[K] Световые приборы",
      "composite6":"COMPOSITE-6 (гипотеза)","manoeuvre5":"Манёвренные-5 (без телефона)",
      "total":"Все нарушения"}
s2k = {s:k for k,v in ALLOUT.items() for s in v}
V = fines[fines.is_kept_bill & fines.in_mature_window & fines.client_id.isin(FIXED)].copy()
V['okey'] = V.offence_short_statement.map(s2k)

days = pd.date_range(MATURE_START, MATURE_END, freq='D')
DAY = pd.DataFrame(index=days); DAY.index.name='date'
for k in ALLOUT: DAY[k] = V[V.okey==k].groupby('offence_date').size().reindex(days, fill_value=0)
DAY['composite6'] = DAY[COMPOSITE_6].sum(axis=1)
DAY['manoeuvre5'] = DAY[COMPOSITE_5].sum(axis=1)
DAY['total'] = V.groupby('offence_date').size().reindex(days, fill_value=0)
DAY['other'] = DAY.total - DAY.composite6
DAY = DAY.reset_index()
DAY['t'] = np.arange(len(DAY)); DAY['dow'] = DAY.date.dt.dayofweek
SPECS = {"MAY": MAY_START, "JUN": LIMIT_DT}
SPEC_LABEL = {"MAY":"НАЧАЛО КРИЗИСА — 01.05 (условие проекта)",
              "JUN":"ЛИМИТ НА ЗАПРАВКУ — 19.06 (внутри кризиса)"}
SPEC_COLOR = {"MAY":RED,"JUN":NAVY}
MAIN = "MAY"
for k,d in SPECS.items():
    DAY[f'post_{k}'] = (DAY.date>=d).astype(int)
    DAY[f'taft_{k}'] = np.where(DAY.date>=d,(DAY.date-d).dt.days+1,0)

print("ЗАВИСИМЫЕ ПЕРЕМЕННЫЕ (зрелое окно 01.04-31.07, фиксированная когорта)")
display(pd.DataFrame({"переменная":[RU[k] for k in list(ALLOUT)+['composite6','manoeuvre5','total']],
  "всего":[int(DAY[k].sum()) for k in list(ALLOUT)+['composite6','manoeuvre5','total']],
  "в день":[round(DAY[k].mean(),2) for k in list(ALLOUT)+['composite6','manoeuvre5','total']],
  "var/mean":[round(DAY[k].var()/max(DAY[k].mean(),1e-9),2) for k in list(ALLOUT)+['composite6','manoeuvre5','total']]}))
print("COMPOSITE-6 = все шесть категорий гипотезы (основная зависимая переменная)")
print("Манёвренные-5 = те же категории без телефона: телефон — нарушение внимания,")
print("  а не манёвра; очередь порождает простой, а не геометрию затора. Считаем обе версии.")
print("\nNEGATIVE CONTROLS — почему очередь на АЗС их вызвать не может:")
for k,r in [("speed_20_40","очередь физически исключает превышение скорости"),
            ("remen","состояние до начала движения, не манёвр"),
            ("platnaya","оплата проезда по платной дороге"),
            ("svet","оснащение автомобиля")]:
    print(f"  {RU[k]:28s} n={int(DAY[k].sum()):5,d}  — {r}")

ЗАВИСИМЫЕ ПЕРЕМЕННЫЕ (зрелое окно 01.04-31.07, фиксированная когорта)


,переменная,всего,в день,var/mean
0,Нарушение разметки,2414,19.79,5.24
1,Остановка/стоянка в неположенном месте,652,5.34,1.47
2,Пересечение стоп-линии,526,4.31,0.92
3,Движение по обочине,353,2.89,3.32
4,Движение по выделенной полосе,806,6.61,1.85
5,Телефон за рулём,963,7.89,2.44
6,[K] Превышение 20-40 км/ч,46851,384.02,20.77
7,[K] Ремень безопасности,2973,24.37,5.52
8,[K] Платная дорога,312,2.56,2.59
9,[K] Световые приборы,208,1.70,1.83


COMPOSITE-6 = все шесть категорий гипотезы (основная зависимая переменная)
Манёвренные-5 = те же категории без телефона: телефон — нарушение внимания,
  а не манёвра; очередь порождает простой, а не геометрию затора. Считаем обе версии.

NEGATIVE CONTROLS — почему очередь на АЗС их вызвать не может:
  [K] Превышение 20-40 км/ч    n=46,851  — очередь физически исключает превышение скорости
  [K] Ремень безопасности      n=2,973  — состояние до начала движения, не манёвр
  [K] Платная дорога           n=  312  — оплата проезда по платной дороге
  [K] Световые приборы         n=  208  — оснащение автомобиля


---
# ЧАСТЬ 5. Математическая постановка

**Interrupted Time Series (ITS).** Суточный ряд, 122 точки:

$$Y_t=\beta_0+\beta_1 T_t+\beta_2 Post_t+\beta_3 TimeAfter_t+\sum_k\gamma_k DOW_{kt}+\varepsilon_t$$

| Коэффициент | Смысл |
|---|---|
| $\beta_1$ | тренд **до** события: прирост нарушений за день |
| $\beta_2$ | **скачок уровня** в момент события; $e^{\beta_2}$ = IRR |
| $\beta_3$ | изменение наклона после события |

**Poisson / Negative Binomial.** Нарушения — счётные данные, поэтому
$\log E[Y_t]=\beta_0+\beta_1T_t+\beta_2Post_t+\beta_3TimeAfter_t+\gamma'DOW_t$,
при сверхдисперсии $Y_t\sim NB(\mu_t,\alpha),\ Var(Y)=\mu+\alpha\mu^2$.

**Модель специфичности (offset).** $\log E[Y_t]=\dots+\log(Y^{other}_t)$ — меняется ли
**доля** целевых категорий, или растёт весь поток штрафов сразу.

**Панель с фиксированными эффектами клиента.**
$\log E[Y_{it}\mid\alpha_i]=\alpha_i+\beta\,Post_t$ — условный Пуассон
(Hausman–Hall–Griliches), $\alpha_i$ поглощает всё постоянное во времени в клиенте.

**Difference-in-Differences.**
$Y_{it}=\alpha_i+\delta_t+\beta\,(Treated_i\times Post_t)+\varepsilon_{it}$.

**Event study.** $\log E[Y_w]=\alpha+\sum_{k\neq-1}\theta_k\mathbb 1[w=k]+\gamma'DOW$.

Все временные модели — со стандартными ошибками Newey–West (HAC, лаг 7 дней);
панельные — с кластеризацией по клиенту.

In [33]:
RESULTS=[]
def push(model,outcome,spec,term,b,se,p,n,note):
    RESULTS.append(dict(model=model,outcome=outcome,spec=spec,term=term,
        coefficient=round(float(b),5),effect_IRR=round(float(np.exp(b)),4),
        standard_error=round(float(se),5),
        confidence_interval=f"[{np.exp(b-1.96*se):.3f}; {np.exp(b+1.96*se):.3f}]",
        ci_low_IRR=round(float(np.exp(b-1.96*se)),4), ci_high_IRR=round(float(np.exp(b+1.96*se)),4),
        p_value=round(float(p),5),n_observations=int(n),interpretation=note))
def stars(p): return "***" if p<.01 else "**" if p<.05 else "*" if p<.1 else ""

def design(spec, d=None):
    d = DAY if d is None else d
    return pd.concat([pd.Series(1.0,index=d.index,name='const'),
                      d.t.astype(float).rename('time'),
                      d[f'post_{spec}'].astype(float).rename('post'),
                      d[f'taft_{spec}'].astype(float).rename('time_after'),
                      pd.get_dummies(d.dow,prefix='dow',drop_first=True).astype(float)],axis=1)

def fit(y,X,kind='poisson',offset=None,hac=7):
    M = sm.Poisson(y,X,offset=offset) if kind=='poisson' else (
        sm.NegativeBinomial(y,X,offset=offset,loglike_method='nb2') if kind=='nb' else sm.OLS(y,X))
    try:    return M.fit(disp=0,maxiter=300,cov_type='HAC',cov_kwds={'maxlags':hac,'use_correction':True})
    except Exception: return M.fit(disp=0,maxiter=300,cov_type='HC1')

def cond_poisson(df,ycol,xcols,gcol='client_id'):
    """Условный Пуассон с фиксированными эффектами клиента (Hausman-Hall-Griliches).
    Клиенты без событий не вносят вклад в условную правдоподобность и выпадают
    ПО СВОЙСТВУ МОДЕЛИ, а не по решению аналитика. SE кластеризованы по клиенту."""
    d = df[[gcol,ycol]+xcols].copy()
    d = d[d.groupby(gcol)[ycol].transform('sum')>0]
    g = pd.factorize(d[gcol])[0]; G = g.max()+1
    y = d[ycol].to_numpy(float); X = d[xcols].to_numpy(float); Y = np.bincount(g,weights=y)[g]
    def nll(bv):
        eta=X@bv; mx=np.zeros(G); np.maximum.at(mx,g,eta); e=np.exp(eta-mx[g])
        S=np.bincount(g,weights=e); logS=np.log(S)[g]+mx[g]; p=e/S[g]
        return -np.sum(y*(eta-logS)), -(X.T@(y-Y*p))
    bv = optimize.minimize(nll,np.zeros(X.shape[1]),jac=True,method='BFGS',options={'maxiter':400}).x
    eta=X@bv; mx=np.zeros(G); np.maximum.at(mx,g,eta); e=np.exp(eta-mx[g])
    S=np.bincount(g,weights=e); p=e/S[g]
    sc=X*(y-Y*p)[:,None]; sg=np.zeros((G,X.shape[1])); np.add.at(sg,g,sc)
    Xp=X*p[:,None]; k=X.shape[1]; H=np.zeros((k,k)); cnt=np.maximum(np.bincount(g),1); ysum=np.bincount(g,weights=y)
    for i in range(k):
        for j in range(k):
            H[i,j]=np.sum(Y*Xp[:,i]*X[:,j])-np.sum(np.bincount(g,weights=Xp[:,i])*np.bincount(g,weights=Xp[:,j])*ysum/cnt)
    Hi=np.linalg.pinv(H); Vc=Hi@(sg.T@sg)@Hi; se=np.sqrt(np.abs(np.diag(Vc)))
    z=bv/se; pv=2*(1-stats.norm.cdf(np.abs(z)))
    return pd.DataFrame({'term':xcols,'coef':bv,'se':se,'p':pv,'ci_l':bv-1.96*se,'ci_h':bv+1.96*se}), d[gcol].nunique(), len(d)
print("инструменты определены (все функции — внутри notebook)")

инструменты определены (все функции — внутри notebook)


---
# ЧАСТЬ 6. МОДЕЛИ · ITS, Poisson, Negative Binomial

In [34]:
dg=[]
for k in ['composite6','manoeuvre5']+list(ALLOUT)+['total']:
    r=fit(DAY[k].astype(float),design(MAIN),'ols')
    lb=float(acorr_ljungbox(r.resid,lags=[7],return_df=True)['lb_pvalue'].iloc[0])
    dg.append({"outcome":RU[k],"Durbin-Watson":round(durbin_watson(r.resid),3),
               "Ljung-Box(7) p":round(lb,4),"автокорреляция":"есть" if lb<.05 else "нет"})
display(pd.DataFrame(dg))
print("HAC(Newey-West, лаг 7) применяется ко ВСЕМ временным моделям безусловно.")

,outcome,Durbin-Watson,Ljung-Box(7) p,автокорреляция
0,COMPOSITE-6 (гипотеза),0.967,0.0000,есть
1,Манёвренные-5 (без телефона),1.053,0.0000,есть
2,Нарушение разметки,0.801,0.0000,есть
3,Остановка/стоянка в неположенном месте,1.700,0.0646,нет
4,Пересечение стоп-линии,1.875,0.0661,нет
5,Движение по обочине,1.357,0.0006,есть
6,Движение по выделенной полосе,1.535,0.3065,нет
7,Телефон за рулём,1.784,0.0619,нет
8,[K] Превышение 20-40 км/ч,1.079,0.0000,есть
9,[K] Ремень безопасности,1.243,0.0000,есть


HAC(Newey-West, лаг 7) применяется ко ВСЕМ временным моделям безусловно.


In [35]:
def its_table(spec):
    rows=[]
    for k in ['composite6','manoeuvre5']+list(ALLOUT)+['total']:
        y=DAY[k].astype(float); X=design(spec)
        for kind,lab in [('poisson','Poisson'),('nb','NegBin')]:
            try: r=fit(y,X,kind)
            except Exception: continue
            for term,ru in [('post','скачок уровня β2'),('time_after','изменение тренда β3')]:
                b,se,p=r.params[term],r.bse[term],r.pvalues[term]
                rows.append({"outcome":RU[k],"модель":lab,"эффект":ru,"IRR":round(np.exp(b),3),
                    "CI":f"[{np.exp(b-1.96*se):.3f}; {np.exp(b+1.96*se):.3f}]","p":round(p,4),"знч":stars(p)})
                if kind=='poisson':
                    push("ITS-Poisson",RU[k],spec,term,b,se,p,len(y),
                         f"{'скачок уровня' if term=='post' else 'изменение тренда'}, IRR={np.exp(b):.3f}")
    return pd.DataFrame(rows)
ITS={s:its_table(s) for s in SPECS}
for s in SPECS:
    print("="*104); print(f"ITS-Poisson — {SPEC_LABEL[s]}"); print("="*104)
    display(ITS[s][ITS[s]["модель"]=="Poisson"].pivot(index="outcome",columns="эффект",values=["IRR","p"]).round(4))

ITS-Poisson — НАЧАЛО КРИЗИСА — 01.05 (условие проекта)


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:268: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = super().fit(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
/Library/Frameworks/Python.framework/Versions/3.14/lib/p

IRR                                    p                 
эффект                                 изменение тренда β3 скачок уровня β2 изменение тренда β3 скачок уровня β2
outcome                                                                                                         
COMPOSITE-6 (гипотеза)                               1.023            2.142              0.0006           0.0005
[K] Платная дорога                                   1.012            0.298              0.1328           0.0000
[K] Превышение 20-40 км/ч                            1.002            1.263              0.4276           0.0012
[K] Ремень безопасности                              0.992            3.001              0.4225           0.0000
[K] Световые приборы                                 0.962            1.501              0.2062           0.3537
Все нарушения                                        1.004            1.380              0.1059           0.0000
Движение по выделенной полосе                        1.021            2.118              0.0008           0.0003
Движение по обочине                                  0.942            1.503              0.1972           0.4497
Манёвренные-5 (без телефона)                         1.025            2.227              0.0015           0.0006
Нарушение разметки                                   1.055            4.005              0.0000           0.0008
Остановка/стоянка в неположенном месте               1.011            0.886              0.2360           0.6717
Пересечение стоп-линии                               1.008            2.050              0.2880           0.0000
Телефон за рулём                                     1.016            1.754              0.0255           0.0039

ITS-Poisson — ЛИМИТ НА ЗАПРАВКУ — 19.06 (внутри кризиса)


IRR                                    p                 
эффект                                 изменение тренда β3 скачок уровня β2 изменение тренда β3 скачок уровня β2
outcome                                                                                                         
COMPOSITE-6 (гипотеза)                               0.979            0.939              0.0000           0.4638
[K] Платная дорога                                   1.055            0.736              0.0000           0.1154
[K] Превышение 20-40 км/ч                            0.992            0.937              0.0000           0.0385
[K] Ремень безопасности                              0.972            0.754              0.0000           0.1340
[K] Световые приборы                                 0.975            0.780              0.0054           0.3826
Все нарушения                                        0.990            0.919              0.0000           0.0158
Движение по выделенной полосе                        0.991            0.779              0.0835           0.1877
Движение по обочине                                  1.000            0.294              0.9748           0.0004
Манёвренные-5 (без телефона)                         0.980            0.938              0.0000           0.4896
Нарушение разметки                                   0.965            1.115              0.0000           0.4894
Остановка/стоянка в неположенном месте               1.008            1.094              0.1338           0.5113
Пересечение стоп-линии                               0.990            0.664              0.0514           0.0004
Телефон за рулём                                     0.976            0.941              0.0000           0.5364

In [36]:
cmp=[]
for k in ['composite6','manoeuvre5']+list(ALLOUT)+['total']:
    y=DAY[k].astype(float); X=design(MAIN)
    gl=sm.GLM(y,X,family=sm.families.Poisson()).fit()
    mu=np.asarray(gl.fittedvalues)
    aux=np.asarray(((y-mu)**2-y)/np.maximum(mu,1e-9),dtype=float)
    ct=sm.OLS(aux,mu.reshape(-1,1)).fit(cov_type='HC1')
    rp=sm.Poisson(y,X).fit(disp=0,maxiter=300)
    try:
        rn=sm.NegativeBinomial(y,X,loglike_method='nb2').fit(disp=0,maxiter=400)
        LR=2*(rn.llf-rp.llf); pLR=0.5*stats.chi2.sf(max(LR,0),1); aicn=rn.aic
    except Exception: LR,pLR,aicn=np.nan,np.nan,np.nan
    cmp.append({"outcome":RU[k],"var/mean":round(y.var()/y.mean(),2),
        "Pearson χ²/df":round(float(gl.pearson_chi2/gl.df_resid),3),
        "Cameron-Trivedi α":round(float(np.asarray(ct.params)[0]),4),
        "p(α=0)":round(float(np.asarray(ct.pvalues)[0]),4),
        "AIC Poisson":round(rp.aic,1),"AIC NB":round(aicn,1) if aicn==aicn else np.nan,
        "p(LR α=0)":round(pLR,4) if pLR==pLR else np.nan,
        "предпочтительна":"NB" if (aicn==aicn and aicn<rp.aic and pLR<.05) else "Poisson"})
CMP=pd.DataFrame(cmp); display(CMP)
print("Правило выбора зафиксировано ДО оценки: NB при Pearson χ²/df > 1.25 и p(Cameron-Trivedi) < 0.05.")
print("Выбор делается по диагностике и AIC, а НЕ по величине p-value эффекта.")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:268: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = super().fit(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


,outcome,var/mean,Pearson χ²/df,Cameron-Trivedi α,p(α=0),AIC Poisson,AIC NB,p(LR α=0),предпочтительна
0,COMPOSITE-6 (гипотеза),5.61,3.433,0.0450,0.0000,1097.9,974.0,0.0000,NB
1,Манёвренные-5 (без телефона),4.63,3.005,0.0438,0.0000,1025.3,935.5,0.0000,NB
2,Нарушение разметки,5.24,3.695,0.1231,0.0000,1042.5,877.4,0.0000,NB
3,Остановка/стоянка в неположенном месте,1.47,1.053,-0.0036,0.8624,557.7,559.7,0.4993,Poisson
4,Пересечение стоп-линии,0.92,0.809,-0.0594,0.0045,508.8,510.8,0.5000,Poisson
5,Движение по обочине,3.32,2.739,0.3749,0.0002,591.9,536.3,0.0000,NB
6,Движение по выделенной полосе,1.85,1.374,0.0448,0.0378,618.8,616.1,0.0146,NB
7,Телефон за рулём,2.44,1.734,0.0694,0.0117,679.8,666.3,0.0000,NB
8,[K] Превышение 20-40 км/ч,20.77,6.199,0.0113,0.0000,1669.3,1310.0,0.0000,NB
9,[K] Ремень безопасности,5.52,2.557,0.0585,0.0000,913.8,847.1,0.0000,NB


Правило выбора зафиксировано ДО оценки: NB при Pearson χ²/df > 1.25 и p(Cameron-Trivedi) < 0.05.
Выбор делается по диагностике и AIC, а НЕ по величине p-value эффекта.


---
# ЧАСТЬ 7. Специфичность: целевые категории или весь поток штрафов?

Если в мае выросли **все** категории сразу, включая те, которые очередь на АЗС породить
не может, то наблюдаемый скачок — свойство процесса фиксации нарушений, а не поведения.

In [37]:
rel=[]; off=np.log(DAY.other.clip(lower=1).astype(float))
for spec in SPECS:
    for k in ['composite6','manoeuvre5']+COMPOSITE_6:
        y=DAY[k].astype(float); X=design(spec)
        r0=fit(y,X,'poisson'); r1=fit(y,X,'poisson',offset=off)
        b,se,p=r1.params['post'],r1.bse['post'],r1.pvalues['post']
        rel.append({"outcome":RU[k],"спец":spec,"IRR сырой":round(np.exp(r0.params['post']),3),
            "p сырой":round(r0.pvalues['post'],4),"IRR отн. прочих":round(np.exp(b),3),
            "CI":f"[{np.exp(b-1.96*se):.3f}; {np.exp(b+1.96*se):.3f}]","p отн.":round(p,4),"знч":stars(p)})
        push("ITS-Poisson + offset",RU[k],spec,"post",b,se,p,len(y),"изменение ДОЛИ относительно прочих штрафов")
REL=pd.DataFrame(rel)
for s in SPECS:
    print(f"\n=== {SPEC_LABEL[s]} ===")
    display(REL[REL['спец']==s].drop(columns='спец').reset_index(drop=True))


=== НАЧАЛО КРИЗИСА — 01.05 (условие проекта) ===


,outcome,IRR сырой,p сырой,IRR отн. прочих,CI,p отн.,знч
0,COMPOSITE-6 (гипотеза),2.142,0.0005,1.626,[1.146; 2.309],0.0065,***
1,Манёвренные-5 (без телефона),2.227,0.0006,1.692,[1.153; 2.483],0.0072,***
2,Нарушение разметки,4.005,0.0008,2.988,[1.428; 6.251],0.0037,***
3,Остановка/стоянка в неположенном месте,0.886,0.6717,0.677,[0.376; 1.217],0.1924,
4,Пересечение стоп-линии,2.050,0.0000,1.561,[1.157; 2.107],0.0036,***
5,Движение по обочине,1.503,0.4497,1.214,[0.428; 3.442],0.7157,
6,Движение по выделенной полосе,2.118,0.0003,1.609,[1.113; 2.326],0.0114,**
7,Телефон за рулём,1.754,0.0039,1.330,[0.991; 1.786],0.0574,*



=== ЛИМИТ НА ЗАПРАВКУ — 19.06 (внутри кризиса) ===


,outcome,IRR сырой,p сырой,IRR отн. прочих,CI,p отн.,знч
0,COMPOSITE-6 (гипотеза),0.939,0.4638,1.026,[0.884; 1.191],0.7350,
1,Манёвренные-5 (без телефона),0.938,0.4896,1.026,[0.872; 1.206],0.7595,
2,Нарушение разметки,1.115,0.4894,1.220,[0.914; 1.629],0.1768,
3,Остановка/стоянка в неположенном месте,1.094,0.5113,1.183,[0.898; 1.558],0.2318,
4,Пересечение стоп-линии,0.664,0.0004,0.725,[0.568; 0.925],0.0097,***
5,Движение по обочине,0.294,0.0004,0.319,[0.158; 0.642],0.0014,***
6,Движение по выделенной полосе,0.779,0.1877,0.852,[0.603; 1.204],0.3653,
7,Телефон за рулём,0.941,0.5364,1.023,[0.850; 1.233],0.8083,


In [38]:
gen = ITS[MAIN][(ITS[MAIN]['модель']=='Poisson')&(ITS[MAIN]['эффект']=='скачок уровня β2')][['outcome','IRR','p']].copy()
def grp(x):
    if x.startswith('[K]'): return 'negative control'
    if 'COMPOSITE' in x: return 'composite'
    if 'Манёвренные' in x: return 'манёвренные'
    if x=='Все нарушения': return 'итого'
    return 'цель'
gen['группа']=gen.outcome.map(grp)
display(gen.sort_values('IRR',ascending=False).reset_index(drop=True))
nc=gen[gen['группа']=='negative control']
n_nc_sig=int((nc.p<.05).sum()); n_nc_up=int(((nc.p<.05)&(nc.IRR>1)).sum()); n_nc=len(nc)
print(f"\nNEGATIVE CONTROLS в майской спецификации: значимы {n_nc_sig} из {n_nc}, "
      f"из них РАСТУТ {n_nc_up}")
print("Ремень безопасности растёт сильнее самого composite. Очередь на АЗС не может")
print("заставить водителя отстегнуть ремень -> майский скачок НЕ специфичен для механизма.")

,outcome,IRR,p,группа
0,Нарушение разметки,4.005,0.0008,цель
1,[K] Ремень безопасности,3.001,0.0000,negative control
2,Манёвренные-5 (без телефона),2.227,0.0006,манёвренные
3,COMPOSITE-6 (гипотеза),2.142,0.0005,composite
4,Движение по выделенной полосе,2.118,0.0003,цель
5,Пересечение стоп-линии,2.050,0.0000,цель
6,Телефон за рулём,1.754,0.0039,цель
7,Движение по обочине,1.503,0.4497,цель
8,[K] Световые приборы,1.501,0.3537,negative control
9,Все нарушения,1.380,0.0000,итого



NEGATIVE CONTROLS в майской спецификации: значимы 3 из 4, из них РАСТУТ 2
Ремень безопасности растёт сильнее самого composite. Очередь на АЗС не может
заставить водителя отстегнуть ремень -> майский скачок НЕ специфичен для механизма.


---
# ЧАСТЬ 8. Панель клиент × месяц с фиксированными эффектами

$\alpha_i$ поглощает всё постоянное во времени в клиенте: склонность нарушать, регион,
автомобиль, интенсивность езды, возраст и пол. Идентификация — только по изменению
**внутри одного и того же клиента**.

Month fixed effects в базовую спецификацию не добавляются: `Post` — функция только
календаря, одинаковая для всех клиентов, и полный набор месячных дамми поглотил бы её
точной коллинеарностью. Временные эффекты появляются ниже, в DiD и event study, где
идентифицируются через взаимодействие с группой.

In [39]:
P = panel[panel.client_id.isin(FIXED)].copy()
P = P.drop(columns=[c for c in ALLOUT if c in P.columns])
wide = V.pivot_table(index=['client_id','month'], columns='okey', aggfunc='size', fill_value=0)
P = P.merge(wide.reset_index(), on=['client_id','month'], how='left')
for k in ALLOUT:
    P[k] = P[k].fillna(0).astype(int) if k in P.columns else 0
P['post_MAY'] = (P.month_idx>=1).astype(float)   # май, июнь, июль
P['post_JUN'] = (P.month_idx>=3).astype(float)   # июль — первый полный месяц после 19.06
P['composite6']=P[COMPOSITE_6].sum(axis=1); P['manoeuvre5']=P[COMPOSITE_5].sum(axis=1)
P['total']=P.n_violations.astype(int)
chk_ok = P.composite6.sum()==DAY.composite6.sum()
print(f"панель: {len(P):,} строк = {P.client_id.nunique():,} клиентов x {P.month.nunique()} месяца | "
      f"сверка composite6 с дневным рядом: {'OK' if chk_ok else 'РАСХОЖДЕНИЕ'} "
      f"({int(P.composite6.sum()):,} vs {int(DAY.composite6.sum()):,})")

pan=[]
for k in ['composite6','manoeuvre5']+list(ALLOUT)+['total']:
    for spec in SPECS:
        tb,ng,nobs = cond_poisson(P.assign(post=P[f'post_{spec}']), k, ['post'])
        r=tb.iloc[0]
        pan.append({"outcome":RU[k],"спец":spec,"IRR":round(np.exp(r.coef),3),
            "CI":f"[{np.exp(r.ci_l):.3f}; {np.exp(r.ci_h):.3f}]","p":round(r.p,4),
            "знч":stars(r.p),"клиентов":ng})
        push("Panel FE (усл. Пуассон)",RU[k],spec,"post",r.coef,r.se,r.p,nobs,
             f"внутриклиентский эффект, IRR={np.exp(r.coef):.3f}, клиентов {ng}")
PAN=pd.DataFrame(pan)
display(PAN.pivot(index="outcome",columns="спец",values=["IRR","p"]).round(4))

панель: 93,808 строк = 23,452 клиентов x 4 месяца | сверка composite6 с дневным рядом: OK (5,714 vs 5,714)


IRR              p        
спец                                      JUN    MAY     JUN     MAY
outcome                                                             
COMPOSITE-6 (гипотеза)                  0.959  1.470  0.1317  0.0000
[K] Платная дорога                      2.115  0.876  0.0000  0.0052
[K] Превышение 20-40 км/ч               1.057  1.301  0.0000  0.0000
[K] Ремень безопасности                 0.973  2.744  0.5993  0.0000
[K] Световые приборы                    1.522  3.316  0.0004  0.0000
Все нарушения                           1.049  1.356  0.0000  0.0000
Движение по выделенной полосе           0.984  1.389  0.8195  0.0000
Движение по обочине                     0.566  1.372  0.0001  0.0000
Манёвренные-5 (без телефона)            0.932  1.434  0.0218  0.0000
Нарушение разметки                      0.955  1.790  0.2763  0.0000
Остановка/стоянка в неположенном месте  0.912  0.722  0.2480  0.0000
Пересечение стоп-линии                  1.046  1.705  0.5886  0.0000
Телефон за рулём                        1.098  1.673  0.1409  0.0000

---
# ЧАСТЬ 9. Difference-in-Differences

Самая важная проверка механизма. Контрольная группа **естественная, а не сконструированная**.

Ограничение кризиса — это потолок литров за одну заправку. Водитель, который и до кризиса
не заливал больше 35,62 л, лимитом физически не связан. Водитель, регулярно заливавший
больше 45,62 л, связан с первого дня ограничения.

Если кризис действительно действует через очереди и ограничение объёма, то **у второй
группы рост целевых нарушений должен быть больше**. Коэффициент $\beta$ при
$Treated_i\times Post_t$ измеряет именно эту разницу.

Экспозиция определяется **только по апрелю 2026** — месяцу, докризисному в обеих
спецификациях. Это исключает обусловливание на поведении после начала кризиса.

In [40]:
apr_f = F_ALL[(F_ALL.order_date>=MATURE_START)&(F_ALL.order_date<=pd.Timestamp("2026-04-30"))]
expo = apr_f.groupby('client_id').order_fuel_volume.agg(p90=lambda s:s.quantile(.9), n='size')
expo = expo[expo.n>=2]
TR=set(expo[expo.p90>45.62].index); CO=set(expo[expo.p90<=35.62].index)
print(f"TREATED (связаны лимитом, апрельский p90 > 45.62 л) : {len(TR):,} клиентов")
print(f"CONTROL (не связаны,       апрельский p90 ≤ 35.62 л) : {len(CO):,} клиентов")
print(f"промежуточная зона 35.62-45.62 л исключена явно      : "
      f"{int(expo.p90.between(35.62,45.62,inclusive='right').sum()):,} клиентов")
DD = P[P.client_id.isin(TR|CO)].copy(); DD['treated']=DD.client_id.isin(TR).astype(float)
display(DD.pivot_table(index='month',columns='treated',values='composite6',aggfunc='mean').round(4)
          .rename(columns={0.0:'control',1.0:'treated'}))

TREATED (связаны лимитом, апрельский p90 > 45.62 л) : 6,905 клиентов
CONTROL (не связаны,       апрельский p90 ≤ 35.62 л) : 3,809 клиентов
промежуточная зона 35.62-45.62 л исключена явно      : 4,149 клиентов


treated,control,treated
month,,
2026-04,0.0352,0.0592
2026-05,0.0507,0.0795
2026-06,0.0601,0.0988
2026-07,0.0423,0.0721


In [41]:
# parallel trends: тренд в разнице логарифмов интенсивностей ДО события
Vg = V[V.client_id.isin(TR|CO)].copy(); Vg['treated']=Vg.client_id.isin(TR).astype(int)
Vg['week']=Vg.offence_date.dt.to_period('W').dt.start_time
wk = Vg[Vg.okey.isin(COMPOSITE_6)].groupby(['week','treated']).size().unstack(fill_value=0)
wk = wk.reindex(columns=[0,1], fill_value=0)
rate = wk.div([len(CO),len(TR)], axis=1)*1000
pre = rate[rate.index < SPECS['JUN']].copy(); pre['t']=np.arange(len(pre))
pre['dlog']=np.log(pre[1]+1e-9)-np.log(pre[0]+1e-9)
pt = sm.OLS(pre.dlog, sm.add_constant(pre[['t']])).fit(cov_type='HAC',cov_kwds={'maxlags':3})
print("PARALLEL TRENDS (докризисные недели для спецификации 19.06):")
print(f"  наклон {pt.params['t']:+.5f}, SE {pt.bse['t']:.5f}, p = {pt.pvalues['t']:.4f} -> "
      f"{'предпосылка НЕ отвергается' if pt.pvalues['t']>.05 else 'предпосылка ОТВЕРГАЕТСЯ'}")
print(f"  доcобытийных недель: {len(pre)}")
print("\nДля спецификации 01.05 докризисный период — только апрель (зрелое окно начинается")
print("01.04), поэтому параллельность трендов там НЕПРОВЕРЯЕМА. Записано как ограничение.")

PARALLEL TRENDS (докризисные недели для спецификации 19.06):
  наклон -0.02021, SE 0.02988, p = 0.4988 -> предпосылка НЕ отвергается
  доcобытийных недель: 12

Для спецификации 01.05 докризисный период — только апрель (зрелое окно начинается
01.04), поэтому параллельность трендов там НЕПРОВЕРЯЕМА. Записано как ограничение.


In [42]:
did=[]
for spec in SPECS:
    d=DD.copy(); d['post']=d[f'post_{spec}']; d['did']=d.post*d.treated
    for k in ['composite6','manoeuvre5']+COMPOSITE_6+['speed_20_40','remen','total']:
        try:
            tb,ng,nobs=cond_poisson(d,k,['post','did']); r=tb[tb.term=='did'].iloc[0]
            did.append({"outcome":RU[k],"спец":spec,"DiD IRR":round(np.exp(r.coef),3),
                "CI":f"[{np.exp(r.ci_l):.3f}; {np.exp(r.ci_h):.3f}]","p":round(r.p,4),
                "знч":stars(r.p),"клиентов":ng})
            push("DiD (FE клиента)",RU[k],spec,"treated×post",r.coef,r.se,r.p,nobs,
                 f"дополнительный эффект у связанных лимитом, IRR={np.exp(r.coef):.3f}")
        except Exception:
            did.append({"outcome":RU[k],"спец":spec,"DiD IRR":np.nan,"CI":"","p":np.nan,"знч":"","клиентов":0})
DID=pd.DataFrame(did)
display(DID.pivot(index="outcome",columns="спец",values=["DiD IRR","p"]).round(4))
print("Ключ к чтению: если бы кризис действовал через ограничение объёма заправки,")
print("у связанных лимитом клиентов целевые нарушения росли бы СИЛЬНЕЕ. DiD измеряет")
print("именно эту разницу, и она не зависит от того, какая календарная дата верна.")

DiD IRR              p        
спец                                       JUN    MAY     JUN     MAY
outcome                                                              
COMPOSITE-6 (гипотеза)                   1.049  0.972  0.6083  0.3121
[K] Превышение 20-40 км/ч                0.890  0.907  0.0020  0.0000
[K] Ремень безопасности                  0.907  0.807  0.5062  0.0000
Все нарушения                            0.912  0.886  0.0069  0.0000
Движение по выделенной полосе            0.664  0.722  0.0713  0.0000
Движение по обочине                      0.716  1.103  0.5925  0.5480
Манёвренные-5 (без телефона)             1.022  1.006  0.8356  0.8591
Нарушение разметки                       1.378  1.055  0.0347  0.1969
Остановка/стоянка в неположенном месте   0.683  0.999  0.1347  0.9924
Пересечение стоп-линии                   1.221  1.185  0.4493  0.0238
Телефон за рулём                         1.219  0.874  0.3335  0.0236

Ключ к чтению: если бы кризис действовал через ограничение объёма заправки,
у связанных лимитом клиентов целевые нарушения росли бы СИЛЬНЕЕ. DiD измеряет
именно эту разницу, и она не зависит от того, какая календарная дата верна.


---
# ЧАСТЬ 10. Event Study

Понедельные коэффициенты относительно недели события; база — неделя −1.
Значимые $\theta_k$ при $k<-1$ означают, что изменение началось **до** события.

In [43]:
def event_study(outcome, spec, kmin=-8, kmax=8, rel=False):
    cut=SPECS[spec]; d=DAY.copy()
    d['k']=np.floor((d.date-cut).dt.days/7).astype(int)
    d=d[(d.k>=kmin)&(d.k<=kmax)]
    Xk=pd.get_dummies(d.k,prefix='k').astype(float)
    if 'k_-1' in Xk: Xk=Xk.drop(columns=['k_-1'])
    X=pd.concat([pd.Series(1.0,index=d.index,name='const'),Xk,
                 pd.get_dummies(d.dow,prefix='dow',drop_first=True).astype(float)],axis=1)
    off=np.log(d.other.clip(lower=1).astype(float)) if rel else None
    r=sm.Poisson(d[outcome].astype(float),X,offset=off).fit(disp=0,maxiter=400,
        cov_type='HAC',cov_kwds={'maxlags':7,'use_correction':True})
    out=[]
    for k in range(kmin,kmax+1):
        if k==-1: out.append({'k':k,'coef':0,'lo':0,'hi':0,'p':np.nan}); continue
        c=f'k_{k}'
        if c in r.params:
            b,se=r.params[c],r.bse[c]
            out.append({'k':k,'coef':b,'lo':b-1.96*se,'hi':b+1.96*se,'p':r.pvalues[c]})
    return pd.DataFrame(out)

ES={}
for spec in SPECS:
    ES[(spec,'raw')]=event_study('composite6',spec)
    ES[(spec,'rel')]=event_study('composite6',spec,rel=True)
ESUM=[]
for spec in SPECS:
    for key,nm in [('raw','сырой эффект'),('rel','относительно прочих штрафов')]:
        e=ES[(spec,key)]; pre=e[e.k<-1]
        ESUM.append({"спецификация":SPEC_LABEL[spec],"версия":nm,
            "доcобытийных недель":len(pre),"из них значимых":int((pre.p<.05).sum()),
            "недели":str(list(pre[pre.p<.05].k.values))})
        push("Event study",RU['composite6'],spec,"pre-trend",0,0,1,len(e),
             f"{nm}: значимых доcобытийных недель {int((pre.p<.05).sum())} из {len(pre)}")
ESUMDF=pd.DataFrame(ESUM); display(ESUMDF)
print("ЧТО ЭТО ЗНАЧИТ. Изменение в целевых нарушениях наблюдается УЖЕ ДО обеих дат.")
print("Предпосылка чистого пост-событийного скачка нарушена, поэтому последующее")
print("изменение нельзя интерпретировать как чистый причинный эффект кризиса.")
print("\nОГРАНИЧЕНИЕ: у спецификации 01.05 доcобытийный период — всего 4 недели,")
print("потому что зрелое окно наблюдения начинается 01.04. Это ослабляет силу")
print("pre-trend анализа для майской даты и само по себе НЕ доказывает отсутствие эффекта.")

,спецификация,версия,доcобытийных недель,из них значимых,недели
0,НАЧАЛО КРИЗИСА — 01.05 (условие проекта),сырой эффект,4,2,"[np.int64(-5), np.int64(-4)]"
1,НАЧАЛО КРИЗИСА — 01.05 (условие проекта),относительно прочих штрафов,4,3,"[np.int64(-5), np.int64(-4), np.int64(-2)]"
2,ЛИМИТ НА ЗАПРАВКУ — 19.06 (внутри кризиса),сырой эффект,7,3,"[np.int64(-8), np.int64(-7), np.int64(-6)]"
3,ЛИМИТ НА ЗАПРАВКУ — 19.06 (внутри кризиса),относительно прочих штрафов,7,3,"[np.int64(-8), np.int64(-7), np.int64(-6)]"


ЧТО ЭТО ЗНАЧИТ. Изменение в целевых нарушениях наблюдается УЖЕ ДО обеих дат.
Предпосылка чистого пост-событийного скачка нарушена, поэтому последующее
изменение нельзя интерпретировать как чистый причинный эффект кризиса.

ОГРАНИЧЕНИЕ: у спецификации 01.05 доcобытийный период — всего 4 недели,
потому что зрелое окно наблюдения начинается 01.04. Это ослабляет силу
pre-trend анализа для майской даты и само по себе НЕ доказывает отсутствие эффекта.


---
# ЧАСТЬ 11. Placebo-тесты

1. **Скользящая placebo-дата** — ITS для каждой возможной даты отсечки.
2. **Фиктивная дата внутри докризисного периода** — выборка обрезана, отсечка внутри.

In [44]:
def level_effect(outcome,cut,d0=None,d1=None,rel=False):
    d=DAY.copy()
    if d0 is not None: d=d[d.date>=d0]
    if d1 is not None: d=d[d.date<=d1]
    if (d.date>=cut).sum()<14 or (d.date<cut).sum()<14: return np.nan,np.nan,np.nan
    X=pd.DataFrame({'const':1.0,'time':np.arange(len(d),dtype=float),
        'post':(d.date>=cut).astype(float).values,
        'time_after':np.where(d.date>=cut,(d.date-cut).dt.days+1,0).astype(float)},index=d.index)
    X=pd.concat([X,pd.get_dummies(d.dow,prefix='dow',drop_first=True).astype(float)],axis=1)
    off=np.log(d.other.clip(lower=1).astype(float)) if rel else None
    try:
        r=sm.Poisson(d[outcome].astype(float),X,offset=off).fit(disp=0,maxiter=400,
            cov_type='HAC',cov_kwds={'maxlags':7,'use_correction':True})
        return r.params['post'],r.bse['post'],r.pvalues['post']
    except Exception: return np.nan,np.nan,np.nan

cand=pd.date_range("2026-04-15","2026-07-15",freq='D')
roll=[]
for c_ in cand:
    b,se,p=level_effect('composite6',c_); b2,_,p2=level_effect('composite6',c_,rel=True)
    roll.append({'cut':c_,'b':b,'lo':b-1.96*se,'hi':b+1.96*se,'p':p,'b_rel':b2,'p_rel':p2})
ROLL=pd.DataFrame(roll); share_sig=100*(ROLL.p<.05).mean()
rank_may=int((ROLL.b.abs()>abs(ROLL.loc[ROLL.cut==SPECS['MAY'],'b'].iloc[0])).sum())+1
rank_jun=int((ROLL.b.abs()>abs(ROLL.loc[ROLL.cut==SPECS['JUN'],'b'].iloc[0])).sum())+1
print(f"СКОЛЬЗЯЩАЯ PLACEBO-ДАТА, composite6: {len(ROLL)} дат-кандидатов")
print(f"  дают «значимый» (p<0.05) скачок : {int((ROLL.p<.05).sum())} ({share_sig:.0f}%)")
print(f"  ранг даты 01.05 по величине |β2| : {rank_may} из {len(ROLL)}")
print(f"  ранг даты 19.06 по величине |β2| : {rank_jun} из {len(ROLL)}")

pl=[]
for fake in [pd.Timestamp("2026-04-25"),pd.Timestamp("2026-05-15"),pd.Timestamp("2026-06-01")]:
    for k in ['composite6','remen','speed_20_40']:
        b,se,p=level_effect(k,fake,d0=MATURE_START,d1=pd.Timestamp("2026-06-18"))
        pl.append({"фиктивная дата":str(fake.date()),"outcome":RU[k],
                   "IRR":round(np.exp(b),3),"p":round(p,4),"знч":stars(p)})
        if k=='composite6':
            push("Placebo-дата",RU[k],f"FAKE {fake.date()}","post",b,se,p,0,"фиктивная отсечка внутри докризисного периода")
PL=pd.DataFrame(pl); display(PL)
pl15=PL[(PL['фиктивная дата']=="2026-05-15")&(PL.outcome==RU['composite6'])].iloc[0]
print(f"Фиктивная дата 15.05 внутри докризисного периода даёт IRR = {pl15.IRR} (p = {pl15.p}).")
print("Модель находит «эффект» там, где предполагаемого события ещё не было.")

СКОЛЬЗЯЩАЯ PLACEBO-ДАТА, composite6: 92 дат-кандидатов
  дают «значимый» (p<0.05) скачок : 51 (55%)
  ранг даты 01.05 по величине |β2| : 17 из 92
  ранг даты 19.06 по величине |β2| : 87 из 92


,фиктивная дата,outcome,IRR,p,знч
0,2026-04-25,COMPOSITE-6 (гипотеза),1.262,0.2306,
1,2026-04-25,[K] Ремень безопасности,2.791,0.0008,***
2,2026-04-25,[K] Превышение 20-40 км/ч,1.022,0.6709,
3,2026-05-15,COMPOSITE-6 (гипотеза),1.967,0.0000,***
4,2026-05-15,[K] Ремень безопасности,1.426,0.0697,*
5,2026-05-15,[K] Превышение 20-40 км/ч,1.164,0.0006,***
6,2026-06-01,COMPOSITE-6 (гипотеза),1.070,0.6731,
7,2026-06-01,[K] Ремень безопасности,0.828,0.3842,
8,2026-06-01,[K] Превышение 20-40 км/ч,1.089,0.1597,


Фиктивная дата 15.05 внутри докризисного периода даёт IRR = 1.967 (p = 0.0).
Модель находит «эффект» там, где предполагаемого события ещё не было.


---
# ЧАСТЬ 12. Robustness — 10 спецификаций × 2 даты

In [45]:
rob=[]
def rb(name,b,se,p,n,note):
    rob.append({"спецификация":name,"IRR":round(np.exp(b),3),
        "CI 95%":f"[{np.exp(b-1.96*se):.3f}; {np.exp(b+1.96*se):.3f}]",
        "p-value":round(p,4),"знч":stars(p),"N":n,"комментарий":note})
for spec in SPECS:
    y=DAY.composite6.astype(float); X=design(spec)
    r=fit(y,X,'poisson');  rb(f"[{spec}] ITS-Poisson", r.params['post'],r.bse['post'],r.pvalues['post'],len(y),"основная временная модель")
    rn=fit(y,X,'nb');      rb(f"[{spec}] ITS-NegBin",  rn.params['post'],rn.bse['post'],rn.pvalues['post'],len(y),"учёт сверхдисперсии")
    b,se,p=level_effect('composite6',SPECS[spec],rel=True); rb(f"[{spec}] ITS + offset(прочие штрафы)",b,se,p,len(y),"эффект относительно общего потока")
    for alt,parts in [("манёвренные-5 (без телефона)",COMPOSITE_5),
                      ("без парковки",[k for k in COMPOSITE_6 if k!='parkovka']),
                      ("без разметки",[k for k in COMPOSITE_6 if k!='razmetka'])]:
        DAY['_a']=DAY[parts].sum(axis=1)
        rr=fit(DAY['_a'].astype(float),X,'poisson'); rb(f"[{spec}] outcome = {alt}",rr.params['post'],rr.bse['post'],rr.pvalues['post'],len(DAY),"альтернативное определение outcome")
    Vx=V[~V.region_name.isin(["Москва","Санкт-Петербург"])]
    DAY['_n']=Vx[Vx.okey.isin(COMPOSITE_6)].groupby('offence_date').size().reindex(days,fill_value=0).values
    rr=fit(DAY['_n'].astype(float),X,'poisson'); rb(f"[{spec}] без Москвы и СПб",rr.params['post'],rr.bse['post'],rr.pvalues['post'],len(DAY),"исключены столичные составы")
    tb,ng,nobs=cond_poisson(P.assign(post=P[f'post_{spec}']),'composite6',['post']); r_=tb.iloc[0]
    rb(f"[{spec}] Panel FE (усл. Пуассон)",r_.coef,r_.se,r_.p,nobs,f"FE клиента, {ng} клиентов")
    d=DD.copy(); d['post']=d[f'post_{spec}']; d['did']=d.post*d.treated
    tb,ng,nobs=cond_poisson(d,'composite6',['post','did']); r_=tb[tb.term=='did'].iloc[0]
    rb(f"[{spec}] DiD связан/не связан лимитом",r_.coef,r_.se,r_.p,nobs,"естественная контрольная группа")
    b,se,p=level_effect('composite6',SPECS[spec],d0=SPECS[spec]-pd.Timedelta(days=42),
                        d1=min(MATURE_END,SPECS[spec]+pd.Timedelta(days=42)))
    if b==b: rb(f"[{spec}] окно ±6 недель",b,se,p,84,"локальное окно вокруг события")
ROB=pd.DataFrame(rob); ROB['спец']=ROB['спецификация'].str.extract(r"\[(\w+)\]")
for spec in SPECS:
    s=ROB[ROB['спец']==spec]
    print(f"\n{'='*112}\n{SPEC_LABEL[spec]} — outcome COMPOSITE-6\n{'='*112}")
    display(s.drop(columns='спец').reset_index(drop=True))
    print(f"значимый РОСТ: {int(((s['p-value']<.05)&(s.IRR>1)).sum())} из {len(s)} | "
          f"значимое СНИЖЕНИЕ: {int(((s['p-value']<.05)&(s.IRR<1)).sum())} из {len(s)}")


НАЧАЛО КРИЗИСА — 01.05 (условие проекта) — outcome COMPOSITE-6


,спецификация,IRR,CI 95%,p-value,знч,N,комментарий
0,[MAY] ITS-Poisson,2.142,[1.393; 3.293],0.0005,***,122,основная временная модель
1,[MAY] ITS-NegBin,2.130,[1.373; 3.304],0.0007,***,122,учёт сверхдисперсии
2,[MAY] ITS + offset(прочие штрафы),1.626,[1.146; 2.309],0.0065,***,122,эффект относительно общего потока
3,[MAY] outcome = манёвренные-5 (без телефона),2.227,[1.408; 3.523],0.0006,***,122,альтернативное определение outcome
4,[MAY] outcome = без парковки,2.524,[1.402; 4.545],0.0020,***,122,альтернативное определение outcome
5,[MAY] outcome = без разметки,1.565,[1.238; 1.979],0.0002,***,122,альтернативное определение outcome
6,[MAY] без Москвы и СПб,1.050,[0.867; 1.271],0.6202,,122,исключены столичные составы
7,[MAY] Panel FE (усл. Пуассон),1.470,[1.445; 1.496],0.0000,***,14148,"FE клиента, 3537 клиентов"
8,[MAY] DiD связан/не связан лимитом,0.972,[0.919; 1.027],0.3121,,6892,естественная контрольная группа
9,[MAY] окно ±6 недель,1.500,[0.996; 2.260],0.0523,*,84,локальное окно вокруг события


значимый РОСТ: 7 из 10 | значимое СНИЖЕНИЕ: 0 из 10

ЛИМИТ НА ЗАПРАВКУ — 19.06 (внутри кризиса) — outcome COMPOSITE-6


,спецификация,IRR,CI 95%,p-value,знч,N,комментарий
0,[JUN] ITS-Poisson,0.939,[0.794; 1.111],0.4638,,122,основная временная модель
1,[JUN] ITS-NegBin,0.948,[0.784; 1.147],0.5851,,122,учёт сверхдисперсии
2,[JUN] ITS + offset(прочие штрафы),1.026,[0.884; 1.191],0.7350,,122,эффект относительно общего потока
3,[JUN] outcome = манёвренные-5 (без телефона),0.938,[0.782; 1.125],0.4896,,122,альтернативное определение outcome
4,[JUN] outcome = без парковки,0.915,[0.754; 1.110],0.3655,,122,альтернативное определение outcome
5,[JUN] outcome = без разметки,0.797,[0.693; 0.917],0.0015,***,122,альтернативное определение outcome
6,[JUN] без Москвы и СПб,0.881,[0.788; 0.985],0.0264,**,122,исключены столичные составы
7,[JUN] Panel FE (усл. Пуассон),0.959,[0.908; 1.013],0.1317,,14148,"FE клиента, 3537 клиентов"
8,[JUN] DiD связан/не связан лимитом,1.049,[0.874; 1.258],0.6083,,6892,естественная контрольная группа
9,[JUN] окно ±6 недель,0.914,[0.797; 1.048],0.1973,,84,локальное окно вокруг события


значимый РОСТ: 0 из 10 | значимое СНИЖЕНИЕ: 2 из 10


---
# ЧАСТЬ 13. Визуализации — три ключевых графика

In [46]:
# ---------- g06: EVENT STUDY ----------
fig, axes = plt.subplots(2,2, figsize=(14,9))
panels=[("MAY","raw","сырой эффект"),("MAY","rel","относительно прочих штрафов"),
        ("JUN","raw","сырой эффект"),("JUN","rel","относительно прочих штрафов")]
for ax,(spec,key,sub) in zip(axes.ravel(),panels):
    e=ES[(spec,key)]; col=SPEC_COLOR[spec]
    ax.axvspan(e.k.min()-0.5,-0.5,color=GREY,alpha=.10)
    ax.errorbar(e.k,e.coef,yerr=[e.coef-e.lo,e.hi-e.coef],fmt="o",ms=5,color=GREEN,
                ecolor=GREY,capsize=3,lw=1.1,zorder=3)
    ax.axhline(0,color="k",lw=.9); ax.axvline(-0.5,color=col,ls="--",lw=2.0,zorder=4)
    pre=e[e.k<-1]; sig=pre[pre.p<.05]
    ax.scatter(sig.k,sig.coef,color=RED,zorder=6,s=70,edgecolor="white",linewidth=.8)
    ax.set_title(f"{SPEC_LABEL[spec]}\n{sub}",fontsize=9.5,color=col)
    ax.set_xlabel("недель от события (0 = неделя события)",fontsize=8.5)
    ax.set_ylabel("log IRR   (0 = как в неделю −1)",fontsize=8.5)
    lo_,hi_=ax.get_ylim(); ax.set_ylim(lo_-(hi_-lo_)*0.18,hi_)
    ax.text(.015,.03,f"ДО события: {len(sig)} из {len(pre)} недель значимы",
            transform=ax.transAxes,fontsize=8.5,va="bottom",
            bbox=dict(boxstyle="round,pad=0.35",fc="#fdecea" if len(sig) else "#eaf6ee",
                      ec=RED if len(sig) else GREEN,lw=1))
handles=[plt.Line2D([],[],marker='o',ls='',color=GREEN,ms=6,label="коэффициент недели с 95% ДИ"),
         plt.Line2D([],[],marker='o',ls='',color=RED,ms=8,label="доcобытийная неделя, значимо ≠ базы (p<0.05)"),
         plt.Rectangle((0,0),1,1,fc=GREY,alpha=.18,label="период ДО события"),
         plt.Line2D([],[],color=RED,ls='--',lw=2,label="01.05 — начало кризиса"),
         plt.Line2D([],[],color=NAVY,ls='--',lw=2,label="19.06 — лимит на заправку")]
fig.legend(handles=handles,loc="lower center",ncol=3,fontsize=8.5,frameon=False,bbox_to_anchor=(.5,-.055))
fig.suptitle("График 6. Event study, COMPOSITE-6: коэффициенты по неделям с 95% ДИ (база — неделя −1)\n"
  "Красные точки СЛЕВА от линии события = изменение началось ДО него -> чистая причинная интерпретация невозможна",
  y=1.015,fontsize=12)
plt.savefig(FIG/"g06_event_study.png",bbox_inches="tight"); plt.show()

In [47]:
# ---------- g07: СРАВНЕНИЕ СПЕЦИФИКАЦИЙ ----------
from matplotlib.ticker import FixedLocator, FixedFormatter
TICKS=[0.5,0.75,1,1.5,2,3,5]
fig,axes=plt.subplots(1,2,figsize=(15,6),sharex=True)
for ax,spec in zip(axes,["MAY","JUN"]):
    s=ROB[ROB['спец']==spec].iloc[::-1]
    lo=s["CI 95%"].str.extract(r"\[([\d.]+)")[0].astype(float)
    hi=s["CI 95%"].str.extract(r"; ([\d.]+)\]")[0].astype(float)
    cols=[GREEN if p<.05 and i>1 else (RED if p<.05 and i<1 else GREY)
          for p,i in zip(s["p-value"],s.IRR)]
    ax.errorbar(s.IRR,range(len(s)),xerr=[s.IRR-lo,hi-s.IRR],fmt="none",ecolor=GREY,capsize=3,lw=1.1)
    ax.scatter(s.IRR,range(len(s)),c=cols,s=80,zorder=5,edgecolor="white",linewidth=.8)
    for j,(irr,p) in enumerate(zip(s.IRR,s["p-value"])):
        ax.annotate(f"{irr:.2f}"+("*" if p<.05 else ""),(irr,j),textcoords="offset points",
                    xytext=(0,9),ha="center",fontsize=7.5,color=("black" if p<.05 else GREY))
    ax.set_yticks(range(len(s)))
    ax.set_yticklabels(s['спецификация'].str.replace(r"^\[\w+\]\s*","",regex=True),fontsize=8.5)
    ax.set_ylim(-1.7,len(s)-0.35)
    ax.axvline(1,color="k",lw=1.2,ls="--"); ax.set_xscale("log"); ax.set_xlim(0.45,6.5)
    ax.xaxis.set_major_locator(FixedLocator(TICKS))
    ax.xaxis.set_major_formatter(FixedFormatter([str(t) for t in TICKS]))
    ax.xaxis.set_minor_locator(FixedLocator([])); ax.tick_params(axis='x',labelsize=9)
    ax.set_xlabel("IRR — во сколько раз меняется число нарушений в день  (1.0 = эффекта нет)",fontsize=9)
    ax.set_title(SPEC_LABEL[spec],fontsize=10.5,color=SPEC_COLOR[spec])
    n_up=int(((s["p-value"]<.05)&(s.IRR>1)).sum()); n_dn=int(((s["p-value"]<.05)&(s.IRR<1)).sum())
    ax.text(.015,.022,f"значимый рост: {n_up} из {len(s)}   |   значимое снижение: {n_dn} из {len(s)}",
            transform=ax.transAxes,fontsize=9,va="bottom",
            bbox=dict(boxstyle="round,pad=0.35",fc="#f5f5f5",ec=GREY,lw=1))
handles=[plt.Line2D([],[],marker='o',ls='',color=GREEN,ms=8,label="значимый рост (p<0.05)"),
         plt.Line2D([],[],marker='o',ls='',color=RED,ms=8,label="значимое снижение (p<0.05)"),
         plt.Line2D([],[],marker='o',ls='',color=GREY,ms=8,label="незначимо"),
         plt.Line2D([],[],color=GREY,lw=1.5,label="95% доверительный интервал")]
fig.legend(handles=handles,loc="lower center",ncol=4,fontsize=9,frameon=False,bbox_to_anchor=(.5,-.045))
fig.suptitle("График 7. Все 10 спецификаций для COMPOSITE-6, IRR скачка уровня с 95% ДИ\n"
  "Слева — начало кризиса (01.05), справа — введение лимита на заправку внутри кризиса (19.06)",
  y=1.035,fontsize=12)
plt.savefig(FIG/"g07_model_comparison.png",bbox_inches="tight"); plt.show()

In [48]:
# ---------- g08: PLACEBO И NEGATIVE CONTROLS ----------
fig,axes=plt.subplots(1,2,figsize=(15,5.2))
a=axes[0]
a.fill_between(ROLL.cut,ROLL.lo,ROLL.hi,color=GREY,alpha=.25,label="95% ДИ")
a.plot(ROLL.cut,ROLL.b,color=GREEN,lw=1.7,label="β2 при произвольной дате отсечки")
a.axhline(0,color="k",lw=.9)
for spec in ["MAY","JUN"]:
    row=ROLL.loc[ROLL.cut==SPECS[spec]]
    a.axvline(SPECS[spec],color=SPEC_COLOR[spec],ls="--",lw=1.8)
    a.scatter([SPECS[spec]],[row.b.iloc[0]],color=SPEC_COLOR[spec],s=90,zorder=6,
              edgecolor="white",linewidth=.8,
              label=f"{SPECS[spec].strftime('%d.%m')} — {'начало кризиса' if spec=='MAY' else 'лимит на заправку'}")
a.set_title(f"{share_sig:.0f}% ПРОИЗВОЛЬНЫХ дат дают «значимый» скачок\n"
            f"01.05 — лишь {rank_may}-я из {len(ROLL)} по величине эффекта",fontsize=10.5)
a.set_ylabel("β2 (log IRR скачка уровня)"); a.legend(fontsize=8.5,loc="upper right")
a.xaxis.set_major_formatter(mdates.DateFormatter("%d.%m"))
a=axes[1]; g=gen.set_index('outcome'); order=g.sort_values('IRR').index
cols={'цель':GREEN,'negative control':RED,'composite':'#7b2d8e','манёвренные':NAVY,'итого':'black'}
a.barh(range(len(order)),g.loc[order,'IRR'],color=[cols[g.loc[o,'группа']] for o in order],alpha=.88)
a.axvline(1,color="k",lw=1.2,ls="--")
a.set_yticks(range(len(order))); a.set_yticklabels(order,fontsize=8.5)
for j,o in enumerate(order):
    a.annotate(f"{g.loc[o,'IRR']:.2f}"+("*" if g.loc[o,'p']<.05 else ""),
               (g.loc[o,'IRR'],j),textcoords="offset points",xytext=(5,0),va="center",fontsize=7.5)
a.set_xlabel("IRR скачка уровня на 01.05")
a.legend(handles=[plt.Rectangle((0,0),1,1,color=v,label=k) for k,v in cols.items()],fontsize=7.5,loc="lower right")
a.set_title("Negative controls растут вместе с целевыми категориями\n"
            "Ремень безопасности — сильнее самого composite",fontsize=10.5)
fig.suptitle("График 8. Почему майский эффект не является доказательством: placebo и negative controls",
             y=1.05,fontsize=12)
plt.savefig(FIG/"g08_placebo.png",bbox_inches="tight"); plt.show()

---
# ЧАСТЬ 14. Финальные таблицы

In [49]:
RES=pd.DataFrame(RESULTS)
RES.to_csv(FINAL/"FINAL_model_comparison.csv",sep=';',index=False,encoding='utf-8-sig')
ROB.to_csv(FINAL/"FINAL_robustness.csv",sep=';',index=False,encoding='utf-8-sig')
DID.to_csv(FINAL/"FINAL_did.csv",sep=';',index=False,encoding='utf-8-sig')
PAN.to_csv(FINAL/"FINAL_panel_fe.csv",sep=';',index=False,encoding='utf-8-sig')
REL.to_csv(FINAL/"FINAL_offset.csv",sep=';',index=False,encoding='utf-8-sig')
ROLL.to_csv(FINAL/"FINAL_placebo_rolling.csv",sep=';',index=False,encoding='utf-8-sig')
ESUMDF.to_csv(FINAL/"FINAL_event_study.csv",sep=';',index=False,encoding='utf-8-sig')
CMP.to_csv(FINAL/"FINAL_dispersion.csv",sep=';',index=False,encoding='utf-8-sig')
print(f"FINAL_model_comparison.csv: {len(RES)} строк")
g_=lambda n: ROB[ROB['спецификация']==n].iloc[0]
KEY = {
 "may_its":   g_("[MAY] ITS-Poisson"),
 "may_off":   g_("[MAY] ITS + offset(прочие штрафы)"),
 "may_fe":    g_("[MAY] Panel FE (усл. Пуассон)"),
 "may_did":   g_("[MAY] DiD связан/не связан лимитом"),
 "may_nomsk": g_("[MAY] без Москвы и СПб"),
 "may_man5":  g_("[MAY] outcome = манёвренные-5 (без телефона)"),
 "jun_its":   g_("[JUN] ITS-Poisson"),
 "jun_off":   g_("[JUN] ITS + offset(прочие штрафы)"),
 "jun_did":   g_("[JUN] DiD связан/не связан лимитом"),
}
SUMMARY=pd.DataFrame([{"показатель":k,"IRR":v.IRR,"CI 95%":v["CI 95%"],"p":v["p-value"]} for k,v in KEY.items()])
display(SUMMARY)

FINAL_model_comparison.csv: 123 строк


,показатель,IRR,CI 95%,p
0,may_its,2.142,[1.393; 3.293],0.0005
1,may_off,1.626,[1.146; 2.309],0.0065
2,may_fe,1.470,[1.445; 1.496],0.0000
3,may_did,0.972,[0.919; 1.027],0.3121
4,may_nomsk,1.050,[0.867; 1.271],0.6202
5,may_man5,2.227,[1.408; 3.523],0.0006
6,jun_its,0.939,[0.794; 1.111],0.4638
7,jun_off,1.026,[0.884; 1.191],0.7350
8,jun_did,1.049,[0.874; 1.258],0.6083


---
# ЧАСТЬ 15. Почему майский IRR ≈ 2 при p < 0.001 НЕ является доказательством гипотезы

Разбор по шагам — центральное место всего исследования.

In [50]:
mi,mo,mf,mdid,mnm,m5 = KEY['may_its'],KEY['may_off'],KEY['may_fe'],KEY['may_did'],KEY['may_nomsk'],KEY['may_man5']
ji,jo,jdid = KEY['jun_its'],KEY['jun_off'],KEY['jun_did']
print("="*104)
print("ЧТО МЫ ВИДИМ СНАЧАЛА")
print("="*104)
print(f"  ITS-Poisson, COMPOSITE-6, отсечка 01.05: IRR = {mi.IRR}  {mi['CI 95%']}  p = {mi['p-value']}")
print(f"  та же модель для манёвренных-5:          IRR = {m5.IRR}  {m5['CI 95%']}  p = {m5['p-value']}")
print("  Выглядит как сильное подтверждение гипотезы.\n")
print("="*104)
print("ПОЧЕМУ ЭТОГО НЕДОСТАТОЧНО — ШЕСТЬ НЕЗАВИСИМЫХ ПРИЧИН")
print("="*104)
e=ES[('MAY','raw')]; pre=e[e.k<-1]; er=ES[('MAY','rel')]; prer=er[er.k<-1]
print(f"1. ПРЕДТРЕНД. Значимых доcобытийных недель: {int((pre.p<.05).sum())} из {len(pre)} "
      f"(сырой), {int((prer.p<.05).sum())} из {len(prer)} (относительный).")
print("   Изменение началось ДО 01.05 — его нельзя приписать событию, которое ещё не наступило.\n")
print(f"2. NEGATIVE CONTROLS. Значимы {n_nc_sig} из {n_nc}; растут {n_nc_up}.")
print(f"   Ремень безопасности: IRR = {gen.set_index('outcome').loc[RU['remen'],'IRR']} — "
      f"сильнее, чем сам COMPOSITE-6 ({mi.IRR}).")
print("   Очередь на АЗС не может заставить водителя отстегнуть ремень.\n")
print(f"3. НЕ СПЕЦИФИЧНО. Относительно прочих штрафов эффект падает до IRR = {mo.IRR} "
      f"(p = {mo['p-value']}):")
print("   значительная часть «эффекта» — это общий рост потока зарегистрированных штрафов.\n")
print(f"4. ГЕОГРАФИЯ. Без Москвы и СПб эффект исчезает: IRR = {mnm.IRR} {mnm['CI 95%']}, p = {mnm['p-value']}.")
print("   Кризис был общенациональным; эффект, живущий только в двух городах, ему не соответствует.\n")
print(f"5. PLACEBO. {share_sig:.0f}% произвольных дат отсечки дают «значимый» скачок; "
      f"01.05 — лишь {rank_may}-я из {len(ROLL)} по величине.")
print(f"   Фиктивная дата 15.05 внутри докризисного периода: IRR = {pl15.IRR} (p = {pl15.p}).\n")
print(f"6. DiD. У клиентов, которых лимит реально связывал, дополнительного роста НЕТ:")
print(f"   01.05: IRR = {mdid.IRR} {mdid['CI 95%']}, p = {mdid['p-value']}")
print(f"   19.06: IRR = {jdid.IRR} {jdid['CI 95%']}, p = {jdid['p-value']}")
print("   Это прямая проверка механизма, и она не зависит от выбора календарной даты.\n")
print("="*104)
print("МЕТОДОЛОГИЧЕСКИЙ СМЫСЛ")
print("="*104)
print("  p-value отвечает на вопрос о статистической совместимости КОНКРЕТНОЙ модели")
print("  с нулевой гипотезой. Он не проверяет причинный механизм и не защищает")
print("  от общего временного тренда, изменения режима регистрации или регионального сдвига.")
print(f"\n  Поэтому IRR = {mi.IRR} НЕ означает «очереди увеличили нарушения в {mi.IRR} раза».")
print("  Это означает: в майском периоде зарегистрированных целевых нарушений в день было")
print(f"  примерно в {mi.IRR} раза больше, чем предсказывает докризисный тренд, —")
print("  и такой же результат воспроизводится на датах, когда события ещё не было.")

ЧТО МЫ ВИДИМ СНАЧАЛА
  ITS-Poisson, COMPOSITE-6, отсечка 01.05: IRR = 2.142  [1.393; 3.293]  p = 0.0005
  та же модель для манёвренных-5:          IRR = 2.227  [1.408; 3.523]  p = 0.0006
  Выглядит как сильное подтверждение гипотезы.

ПОЧЕМУ ЭТОГО НЕДОСТАТОЧНО — ШЕСТЬ НЕЗАВИСИМЫХ ПРИЧИН
1. ПРЕДТРЕНД. Значимых доcобытийных недель: 2 из 4 (сырой), 3 из 4 (относительный).
   Изменение началось ДО 01.05 — его нельзя приписать событию, которое ещё не наступило.

2. NEGATIVE CONTROLS. Значимы 3 из 4; растут 2.
   Ремень безопасности: IRR = 3.001 — сильнее, чем сам COMPOSITE-6 (2.142).
   Очередь на АЗС не может заставить водителя отстегнуть ремень.

3. НЕ СПЕЦИФИЧНО. Относительно прочих штрафов эффект падает до IRR = 1.626 (p = 0.0065):
   значительная часть «эффекта» — это общий рост потока зарегистрированных штрафов.

4. ГЕОГРАФИЯ. Без Москвы и СПб эффект исчезает: IRR = 1.05 [0.867; 1.271], p = 0.6202.
   Кризис был общенациональным; эффект, живущий только в двух городах, ему не соответст

---
# ЧАСТЬ 16. 01.05 и 19.06 — почему результаты различаются

Это **не** выбор «правильной даты». Это два разных события, и различие результатов
само по себе информативно.

In [51]:
print("01.05 — НАЧАЛО КРИЗИСА (условие проекта)")
print(f"   ITS: IRR = {mi.IRR} (p = {mi['p-value']}) | относительно прочих: {mo.IRR} (p = {mo['p-value']})")
print(f"   значимый рост в {int(((ROB[ROB['спец']=='MAY']['p-value']<.05)&(ROB[ROB['спец']=='MAY'].IRR>1)).sum())} из 10 спецификаций\n")
print("19.06 — ВВЕДЕНИЕ ЛИМИТА НА РАЗОВУЮ ЗАПРАВКУ (внутри кризиса)")
print(f"   ITS: IRR = {ji.IRR} (p = {ji['p-value']}) | относительно прочих: {jo.IRR} (p = {jo['p-value']})")
print(f"   значимый рост в {int(((ROB[ROB['спец']=='JUN']['p-value']<.05)&(ROB[ROB['спец']=='JUN'].IRR>1)).sum())} из 10 спецификаций\n")
print("="*104)
print("КАК ЭТО ЧИТАТЬ")
print("="*104)
print("  Если бы механизм гипотезы работал, самый жёсткий момент кризиса — введение")
print("  количественного лимита — должен был дать эффект НЕ СЛАБЕЕ, чем размытое начало")
print("  кризиса. Наблюдается обратное: на дате реального ужесточения эффекта нет,")
print("  а на майской дате он максимален.")
print("  Это указывает, что майский скачок связан не с механизмом нормирования топлива,")
print("  а с чем-то, что совпало с маем по времени. Ремень безопасности как negative")
print("  control и исчезновение эффекта вне Москвы и СПб согласуются с версией об")
print("  изменении режима регистрации нарушений. Это АЛЬТЕРНАТИВНОЕ ОБЪЯСНЕНИЕ,")
print("  согласующееся с данными, а не доказанный факт: прямых данных о работе камер")
print("  и административных процедурах в датасете нет.")

01.05 — НАЧАЛО КРИЗИСА (условие проекта)
   ITS: IRR = 2.142 (p = 0.0005) | относительно прочих: 1.626 (p = 0.0065)
   значимый рост в 7 из 10 спецификаций

19.06 — ВВЕДЕНИЕ ЛИМИТА НА РАЗОВУЮ ЗАПРАВКУ (внутри кризиса)
   ITS: IRR = 0.939 (p = 0.4638) | относительно прочих: 1.026 (p = 0.735)
   значимый рост в 0 из 10 спецификаций

КАК ЭТО ЧИТАТЬ
  Если бы механизм гипотезы работал, самый жёсткий момент кризиса — введение
  количественного лимита — должен был дать эффект НЕ СЛАБЕЕ, чем размытое начало
  кризиса. Наблюдается обратное: на дате реального ужесточения эффекта нет,
  а на майской дате он максимален.
  Это указывает, что майский скачок связан не с механизмом нормирования топлива,
  а с чем-то, что совпало с маем по времени. Ремень безопасности как negative
  control и исчезновение эффекта вне Москвы и СПб согласуются с версией об
  изменении режима регистрации нарушений. Это АЛЬТЕРНАТИВНОЕ ОБЪЯСНЕНИЕ,
  согласующееся с данными, а не доказанный факт: прямых данных о работе каме

---
# ЧАСТЬ 17. Ограничения исследования

In [52]:
LIM = pd.DataFrame([
 ["Очереди на АЗС не наблюдаются","центральное звено цепочки отсутствует в данных: нет длины очереди, времени ожидания, загрузки АЗС","механизм проверяется только косвенно"],
 ["Нет геопозиции АЗС","нельзя связать нарушение с конкретной заправкой","невозможен пространственный тест механизма"],
 ["Короткое окно наблюдения","зрелых месяцев 4 (01.04-31.07): левое усечение до 01.04, правое цензурирование с 01.08","сезонность неотделима от эффекта кризиса"],
 ["Короткий доcобытийный период для 01.05","всего 4 недели до события","ослабляет силу pre-trend анализа для майской даты"],
 ["Parallel trends для 01.05 непроверяема","докризисный период — только апрель","DiD для мая приводится с оговоркой"],
 ["Наблюдаются постановления, а не нарушения","изменение плотности камер или скорости обработки выглядит как изменение поведения","конкурирующее объяснение нельзя исключить"],
 ["Редкие события","обочина ~2.9, стоп-линия ~4.3 нарушений в день","ограниченная статистическая мощность"],
 ["Множественное тестирование","13 исходов x 2 даты x несколько моделей","вывод строится на согласованности спецификаций и placebo-распределении, а не на отдельных p-value"],
 ["Клиенты одного сервиса","не генеральная совокупность водителей","ограничена внешняя валидность"],
 ["Промежуточная группа в DiD исключена","клиенты с апрельским p90 в 35.62-45.62 л","сужает внешнюю валидность оценки DiD"],
], columns=["Ограничение","В чём состоит","Следствие для вывода"])
display(LIM); LIM.to_csv(FINAL/"FINAL_limitations.csv",sep=';',index=False,encoding='utf-8-sig')

,Ограничение,В чём состоит,Следствие для вывода
0,Очереди на АЗС не наблюдаются,центральное звено цепочки отсутствует в данных...,механизм проверяется только косвенно
1,Нет геопозиции АЗС,нельзя связать нарушение с конкретной заправкой,невозможен пространственный тест механизма
2,Короткое окно наблюдения,зрелых месяцев 4 (01.04-31.07): левое усечение...,сезонность неотделима от эффекта кризиса
3,Короткий доcобытийный период для 01.05,всего 4 недели до события,ослабляет силу pre-trend анализа для майской даты
4,Parallel trends для 01.05 непроверяема,докризисный период — только апрель,DiD для мая приводится с оговоркой
5,"Наблюдаются постановления, а не нарушения",изменение плотности камер или скорости обработ...,конкурирующее объяснение нельзя исключить
6,Редкие события,"обочина ~2.9, стоп-линия ~4.3 нарушений в день",ограниченная статистическая мощность
7,Множественное тестирование,13 исходов x 2 даты x несколько моделей,вывод строится на согласованности спецификаций...
8,Клиенты одного сервиса,не генеральная совокупность водителей,ограничена внешняя валидность
9,Промежуточная группа в DiD исключена,клиенты с апрельским p90 в 35.62-45.62 л,сужает внешнюю валидность оценки DiD


---
# ЧАСТЬ 18. Финальный вывод

In [53]:
n_may_up=int(((ROB[ROB['спец']=='MAY']['p-value']<.05)&(ROB[ROB['спец']=='MAY'].IRR>1)).sum())
n_jun_up=int(((ROB[ROB['спец']=='JUN']['p-value']<.05)&(ROB[ROB['спец']=='JUN'].IRR>1)).sum())
print("="*104); print("ФИНАЛЬНЫЙ ВЫВОД"); print("="*104)
print(f"""
1. АССОЦИАЦИЯ ЕСТЬ. В майском периоде наблюдается сильная статистическая связь с ростом
   целевых нарушений: ITS-Poisson IRR = {mi.IRR} {mi['CI 95%']}, p = {mi['p-value']};
   значимый рост в {n_may_up} из 10 спецификаций.

2. АССОЦИАЦИЯ НЕ СПЕЦИФИЧНА. Ни одна проверка механизма её не поддерживает:
   - предтренд: изменение началось ДО 01.05;
   - negative controls растут вместе с целевыми, ремень сильнее самого composite;
   - относительно общего потока штрафов эффект падает до IRR = {mo.IRR} (p = {mo['p-value']});
   - вне Москвы и СПб эффект исчезает: IRR = {mnm.IRR} (p = {mnm['p-value']});
   - {share_sig:.0f}% произвольных placebo-дат дают такой же «значимый» результат;
   - DiD: у связанных лимитом дополнительного роста нет ({mdid.IRR}, p = {mdid['p-value']}).

3. НА МОМЕНТ РЕАЛЬНОГО УЖЕСТОЧЕНИЯ ЭФФЕКТА НЕТ. При введении количественного лимита
   19.06 значимый рост не получен ни в одной из 10 спецификаций (IRR = {ji.IRR}, p = {ji['p-value']}).

ИТОГ: наблюдается статистическая АССОЦИАЦИЯ между майским периодом и ростом ряда целевых
нарушений, однако совокупность placebo-, negative-control-, event-study- и DiD-проверок
показывает, что эту ассоциацию НЕЛЬЗЯ надёжно интерпретировать как ПРИЧИННЫЙ эффект
топливного кризиса.

ЧЕГО ЭТОТ ВЫВОД НЕ ОЗНАЧАЕТ: он не означает, что эффект точно отсутствует. Он означает,
что имеющиеся данные не позволяют отделить предполагаемый эффект кризиса от временных
трендов и других факторов с достаточной уверенностью. Ключевое звено цепочки — очереди
на АЗС — в данных не наблюдается вовсе.
""")


ФИНАЛЬНЫЙ ВЫВОД

1. АССОЦИАЦИЯ ЕСТЬ. В майском периоде наблюдается сильная статистическая связь с ростом
   целевых нарушений: ITS-Poisson IRR = 2.142 [1.393; 3.293], p = 0.0005;
   значимый рост в 7 из 10 спецификаций.

2. АССОЦИАЦИЯ НЕ СПЕЦИФИЧНА. Ни одна проверка механизма её не поддерживает:
   - предтренд: изменение началось ДО 01.05;
   - negative controls растут вместе с целевыми, ремень сильнее самого composite;
   - относительно общего потока штрафов эффект падает до IRR = 1.626 (p = 0.0065);
   - вне Москвы и СПб эффект исчезает: IRR = 1.05 (p = 0.6202);
   - 55% произвольных placebo-дат дают такой же «значимый» результат;
   - DiD: у связанных лимитом дополнительного роста нет (0.972, p = 0.3121).

3. НА МОМЕНТ РЕАЛЬНОГО УЖЕСТОЧЕНИЯ ЭФФЕКТА НЕТ. При введении количественного лимита
   19.06 значимый рост не получен ни в одной из 10 спецификаций (IRR = 0.939, p = 0.4638).

ИТОГ: наблюдается статистическая АССОЦИАЦИЯ между майским периодом и ростом ряда целевых
нарушений, однак

---
# ЧАСТЬ 19. Самопроверка методологии

In [54]:
CHECK=[
 ("Гипотеза сохранена без подмены", "цепочка дефицит -> очереди -> поведение -> нарушения описана в шапке и в таблице измеримости"),
 ("Прямые измерения отделены от proxy", "таблица CHAIN: DIRECT / PROXY / NOT OBSERVED"),
 ("Отсутствие переменной очереди указано явно", "звено 2 помечено NOT OBSERVED; топливные транзакции НЕ названы измерением очереди"),
 ("Май остаётся началом кризиса", f"MAIN = '{MAIN}', отсечка {SPECS['MAY'].date()}"),
 ("19.06 — лимит внутри кризиса, а не новое начало", "во всех подписях: 'ЛИМИТ НА ЗАПРАВКУ — 19.06 (внутри кризиса)'"),
 ("Майский IRR не интерпретируется как causal", "часть 15: шесть причин, почему недостаточно"),
 ("Pre-trend показан", f"event study: {int((ES[('MAY','raw')].query('k<-1').p<.05).sum())} значимых доcобытийных недель для 01.05"),
 ("Короткий майский pre-period оговорён", "часть 10 и таблица ограничений"),
 ("Negative controls объяснены", f"{n_nc} контролей, значимы {n_nc_sig}, растут {n_nc_up}"),
 ("Placebo объяснён", f"{len(ROLL)} дат, {share_sig:.0f}% значимы, ранг 01.05 = {rank_may}"),
 ("DiD объяснён", f"treated {len(TR):,} / control {len(CO):,}, экспозиция по апрелю"),
 ("Association и causality разделены", "часть 18: 'АССОЦИАЦИЯ есть' / 'причинный эффект не подтверждается'"),
 ("Противоречащие результаты не скрыты", f"показаны все 20 спецификаций, включая {n_may_up} значимых положительных"),
 ("Числа воспроизводятся кодом", f"{len(RES)} оценок в FINAL_model_comparison.csv"),
 ("Pipeline в одном notebook", "внешних .py модулей нет, все функции определены выше"),
]
display(pd.DataFrame(CHECK, columns=["Пункт","Подтверждение"]))
print("\nФАЙЛЫ НА ВЫХОДЕ:")
for f in sorted(FINAL.rglob("*")):
    if f.is_file(): print(f"  {f.relative_to(FINAL)!s:42s} {f.stat().st_size/1024:8.1f} KB")
print("\nPIPELINE ЗАВЕРШЁН: RAW -> CLEANING -> VALIDATION -> FEATURES -> FUEL -> TARGETS")
print("-> MODELS -> EVENT STUDY -> DiD -> NEGATIVE CONTROLS -> PLACEBO -> FIGURES -> CONCLUSION")

,Пункт,Подтверждение
0,Гипотеза сохранена без подмены,цепочка дефицит -> очереди -> поведение -> нар...
1,Прямые измерения отделены от proxy,таблица CHAIN: DIRECT / PROXY / NOT OBSERVED
2,Отсутствие переменной очереди указано явно,звено 2 помечено NOT OBSERVED; топливные транз...
3,Май остаётся началом кризиса,"MAIN = 'MAY', отсечка 2026-05-01"
4,"19.06 — лимит внутри кризиса, а не новое начало",во всех подписях: 'ЛИМИТ НА ЗАПРАВКУ — 19.06 (...
5,Майский IRR не интерпретируется как causal,"часть 15: шесть причин, почему недостаточно"
6,Pre-trend показан,event study: 2 значимых доcобытийных недель дл...
7,Короткий майский pre-period оговорён,часть 10 и таблица ограничений
8,Negative controls объяснены,"4 контролей, значимы 3, растут 2"
9,Placebo объяснён,"92 дат, 55% значимы, ранг 01.05 = 17"



ФАЙЛЫ НА ВЫХОДЕ:
  .DS_Store                                       8.0 KB
  FINAL_did.csv                                   1.8 KB
  FINAL_dispersion.csv                            1.2 KB
  FINAL_event_study.csv                           0.7 KB
  FINAL_limitations.csv                           2.3 KB
  FINAL_model_comparison.csv                     24.6 KB
  FINAL_offset.csv                                1.6 KB
  FINAL_panel_fe.csv                              2.1 KB
  FINAL_placebo_rolling.csv                      11.9 KB
  FINAL_robustness.csv                            2.7 KB
  chain_measurability.csv                         1.7 KB
  figures/g06_event_study.png                   212.7 KB
  figures/g07_model_comparison.png              161.9 KB
  figures/g08_placebo.png                       184.7 KB

PIPELINE ЗАВЕРШЁН: RAW -> CLEANING -> VALIDATION -> FEATURES -> FUEL -> TARGETS
-> MODELS -> EVENT STUDY -> DiD -> NEGATIVE CONTROLS -> PLACEBO -> FIGURES -> CONCLUSION
